# Severity Urgency Rule Framework

This notebook builds a complaint-level urgency score for ODI complaints. The goal here is to help human reviewers decide which complaints should be read first rather than prove if a defect exists or not.

For now, we use `severity_primary_flag` as the target, that flag focuses on the clearest severe signals such as fires, injuries, and deaths. We compare simple structured, text only, and hybrid models, then improve the best one step by step on `valid_2025`.

Working review budgets (the percentage of highest scores we'll pass to reviewers):
- top 1%
- top 2%
- top 5%
- top 10%

Optional output written by this notebook:
- `data/outputs/severity_partner_results.csv`

## 1. Setup

Import the packages, set the repo paths, and make sure the notebook is pointed at the current project folder.


In [75]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from scipy import sparse
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
required_repo_paths = [
    PROJECT_ROOT / 'data',
    PROJECT_ROOT / 'notebooks',
    PROJECT_ROOT / 'README.md'
]
if not all(path.exists() for path in required_repo_paths):
    raise FileNotFoundError(
        f"Run this notebook from the active repo root. Resolved path: {PROJECT_ROOT}"
    )

PROCESSED_DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
OUTPUTS_DIR = PROJECT_ROOT / 'data' / 'outputs'
SEVERITY_PATH = PROCESSED_DATA_DIR / 'odi_severity_cases.parquet'
COMPONENT_MULTI_PATH = PROCESSED_DATA_DIR / 'odi_component_multilabel_cases.parquet'
SEVERITY_RESULTS_PATH = OUTPUTS_DIR / 'severity_partner_results.csv'

RANDOM_SEED = 42
sns.set_theme(style='whitegrid')
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Severity input:", SEVERITY_PATH)
print("Component cases input:", COMPONENT_MULTI_PATH)
print("Exploratory results path:", SEVERITY_RESULTS_PATH)

Project root: C:\Users\davis\OneDrive\Documents\VS Code Repos\NHTSA-ODI-Complaint-Analytics
Severity input: C:\Users\davis\OneDrive\Documents\VS Code Repos\NHTSA-ODI-Complaint-Analytics\data\processed\odi_severity_cases.parquet
Component cases input: C:\Users\davis\OneDrive\Documents\VS Code Repos\NHTSA-ODI-Complaint-Analytics\data\processed\odi_component_multilabel_cases.parquet
Exploratory results path: C:\Users\davis\OneDrive\Documents\VS Code Repos\NHTSA-ODI-Complaint-Analytics\data\outputs\severity_partner_results.csv


## 2. Load The Severity Table

Load the prepared severity table created during cleaning/preprocessing.

In [76]:
if not SEVERITY_PATH.exists():
    raise FileNotFoundError(f"Run preprocessing first. Missing input file: {SEVERITY_PATH}")

df = pd.read_parquet(SEVERITY_PATH)
df['ldate'] = pd.to_datetime(df['ldate'])

df['cdescr_model_text'] = (
    df['cdescr']
    .fillna('')
    .astype(str)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

print("Loaded from:", SEVERITY_PATH)
print("Rows:", len(df))
print("Columns:", len(df.columns))
display(df[['odino', 'ldate', 'mfr_name', 'maketxt', 'modeltxt', 'yeartxt', 'severity_primary_flag', 'severity_broad_flag']].head())

Loaded from: C:\Users\davis\OneDrive\Documents\VS Code Repos\NHTSA-ODI-Complaint-Analytics\data\processed\odi_severity_cases.parquet
Rows: 375937
Columns: 67


,odino,ldate,mfr_name,maketxt,modeltxt,yeartxt,severity_primary_flag,severity_broad_flag
0,10816121,2020-10-05,"GENERAL MOTORS, LLC",PONTIAC,VIBE,2007,False,False
1,11289434,2020-01-02,HONDA (AMERICAN HONDA MOTOR CO.),HONDA,CR-V,2019,False,False
2,11289436,2020-01-02,HYUNDAI MOTOR AMERICA,HYUNDAI,SONATA,2018,True,True
3,11290552,2020-01-02,"CHRYSLER (FCA US, LLC)",DODGE,DURANGO,2011,True,True
4,11292384,2020-01-01,HONDA (AMERICAN HONDA MOTOR CO.),HONDA,ACCORD,2018,False,False


## 3. Choose The Target And Review Budgets

We judge models by how useful they are under a real potential review limit. For this model, we'll make the top 5% of severe cases our review queue and use the top 1%, 2%, and 10% as supporting checks to make sure the model isn't somehow really underperforming in those bands. This way, we can focus on the practical question of "if reviewers can only inspect a small share of complaints, how many truly severe complaints do we catch?"

In [77]:
TARGET_COL = 'severity_primary_flag'
REVIEW_BUDGETS = [0.01, 0.02, 0.05, 0.10]
REVIEW_BUDGET_LABELS = {
    0.01: 'top_1pct',
    0.02: 'top_2pct',
    0.05: 'top_5pct',
    0.10: 'top_10pct'
}

print("Active target:", TARGET_COL)
print("Primary review budget for model choice: top 5%")

target_summary = pd.DataFrame({
    'target': ['severity_primary_flag', 'severity_broad_flag'],
    'positive_cases': [int(df['severity_primary_flag'].sum()), int(df['severity_broad_flag'].sum())],
    'positive_rate': [float(df['severity_primary_flag'].mean()), float(df['severity_broad_flag'].mean())]
})
display(target_summary)

year_summary = df.assign(year=df['ldate'].dt.year).groupby('year').agg(
    rows=('odino', 'count'),
    primary_rate=('severity_primary_flag', 'mean'),
    broad_rate=('severity_broad_flag', 'mean')
).reset_index()
display(year_summary)

Active target: severity_primary_flag
Primary review budget for model choice: top 5%


,target,positive_cases,positive_rate
0,severity_primary_flag,22706,0.060398
1,severity_broad_flag,34071,0.090630


,year,rows,primary_rate,broad_rate
0,2020,58465,0.066843,0.085077
1,2021,49044,0.071568,0.096607
2,2022,53221,0.065425,0.101708
3,2023,62727,0.061887,0.094026
4,2024,67432,0.049976,0.084826
5,2025,73985,0.052565,0.086288
6,2026,11063,0.06011,0.085329


## 4. Make Time-Based Splits

The data is split by year so we follow that time order rather than random sampling (don't want the model to cheat with the power of future sight). We train on complaints through the end of 2024, use 2025 as the validation year, and keep 2026 as a later holdout test check.

In [78]:
train_mask = df['ldate'] <= pd.Timestamp('2024-12-31')
valid_mask = (df['ldate'] >= pd.Timestamp('2025-01-01')) & (df['ldate'] <= pd.Timestamp('2025-12-31'))
holdout_mask = df['ldate'] >= pd.Timestamp('2026-01-01')

train_df = df.loc[train_mask].copy()
valid_df = df.loc[valid_mask].copy()
holdout_df = df.loc[holdout_mask].copy()

y_train = train_df[TARGET_COL].to_numpy()
y_valid = valid_df[TARGET_COL].to_numpy()
y_holdout = holdout_df[TARGET_COL].to_numpy()

split_summary = pd.DataFrame({
    'split': ['train_through_2024', 'valid_2025', 'holdout_2026'],
    'rows': [len(train_df), len(valid_df), len(holdout_df)],
    'positive_rate': [train_df[TARGET_COL].mean(), valid_df[TARGET_COL].mean(), holdout_df[TARGET_COL].mean()]
})
display(split_summary)

,split,rows,positive_rate
0,train_through_2024,290889,0.062402
1,valid_2025,73985,0.052565
2,holdout_2026,11063,0.060110


## 5. Shared Helper Code And Tracking

This notebook is intended to be self-contained so we aren't pulling hidden functions imported from local modules out like a magician. The helper cell below holds the scoring, queue, and comparison code used throughout the analysis. There are some broader metrics in here, but they mainly support the goal of making the top 5% reviews our decision surface.

**This grew over time as I added more experiments and checks. I would personally keep this collapsed and run it so it doesn't take up your screen, unless you want to inspect some of the functions.**

In [79]:
severity_results = []
state_registry = {}

DEFAULT_ALPHA = 1e-5
WORD_NGRAM_RANGE = (1, 2)
CHAR_NGRAM_RANGE = (3, 5)
TEXT_MIN_DF = 5
DEFAULT_WORD_MAX_FEATURES = 20000
DEFAULT_CHAR_MAX_FEATURES = 10000
COMPONENT_GROUP_MIN_TRAIN_CASES = 100
COMPONENT_GROUP_TOP_N = 3
SUSPICIOUS_FLAG_CANDIDATES = [
    'flag_speed_999',
    'flag_speed_high',
    'flag_miles_high',
    'flag_injured_99',
    'flag_deaths_99'
]
URGENCY_BAND_ORDER = ['high_urgency', 'priority_review', 'watch', 'routine']
BASE_REVIEW_COLS = [
    'odino',
    'ldate',
    'mfr_name',
    'maketxt',
    'modeltxt',
    'yeartxt',
    'urgency_score',
    'urgency_band',
    'suspicious_value_any',
    'severity_primary_flag',
    'severity_broad_flag',
    'cdescr_model_text'
]

PRESERVE_STOP_WORDS = {
    'no',
    'not',
    'never',
    'none',
    'without',
    'while',
    'after',
    'before',
    'during',
    'against',
    'off',
    'on',
    'over',
    'under',
    'down',
    'up'
}
CUSTOM_STOP_WORDS = sorted(set(ENGLISH_STOP_WORDS) - PRESERVE_STOP_WORDS)
DOMAIN_TEXT_PATTERNS = [
    (r'(?i)\bair[\s-]?bags\b', ' airbags '),
    (r'(?i)\bair[\s-]?bag\b', ' airbag '),
    (r'(?i)\bcheck[\s-]?engine[\s-]?light\b', ' check_engine_light '),
    (r'(?i)\bpower steering\b', ' power_steering '),
    (r'(?i)\bpower brakes?\b', ' power_brakes '),
    (r'(?i)\bsteering wheel\b', ' steering_wheel '),
    (r'(?i)\bseat[\s-]?belts\b', ' seat_belts '),
    (r'(?i)\bseat[\s-]?belt\b', ' seat_belt '),
    (r'(?i)\bfuel pump\b', ' fuel_pump '),
    (r'(?i)\bwithout warning\b', ' without_warning '),
    (r'(?i)\bwhile driving\b', ' while_driving '),
    (r'(?i)\b(?:loss of power|lost power|lose power|loses power|no power)\b', ' power_loss '),
    (r'(?i)\b(?:would not accelerate|wouldn\'t accelerate|will not accelerate|won\'t accelerate|unable to accelerate|cannot accelerate|can\'t accelerate|failed to accelerate|fails to accelerate|did not accelerate|does not accelerate)\b', ' no_acceleration '),
    (r'(?i)\b(?:stalled out|stalling|stalled|engine died|vehicle died|shut off|shuts off|turned off|turns off)\b', ' stall_shutdown ')
]

ERROR_TEXT_FLAG_PATTERNS = [
    ('has_contact_boilerplate', r'(?i)^\s*(?:the contact|consumer|owner|driver|complainant)\s+(?:states?|stated|owns|owned|leased)\b'),
    ('has_redaction_artifact', r'(?i)\[\s*x+\s*\]|information redacted pursuant to the freedom of information act \(foia\) 5 u\.s\.c\. 552\(b\)\(6\) and 49 c\.f\.r\. 512\.8'),
    ('has_url_or_email', r'(?i)(?:https?://\S+|www\.\S+|[a-z0-9._%+-]+@[a-z0-9.-]+\.[a-z]{2,})'),
    ('has_phone_like', r'(?i)(?:\+?1[-.\s]*)?(?:\(?\d{3}\)?[-.\s]*)\d{3}[-.\s]*\d{4}'),
    ('has_vin_like', r'(?i)\b[a-hj-npr-z0-9]{17}\b'),
    ('has_case_or_claim_id', r'(?i)\b(?:case|claim|ticket|reference|tracking|repair order|work order|invoice)\s*(?:number|no|#)?\s*[:#-]?\s*[a-z0-9-]{5,}\b'),
    ('has_long_mixed_id', r'(?i)\b(?=[a-z0-9-]{8,}\b)(?=[a-z0-9-]*[a-z])(?=[a-z0-9-]*\d)[a-z0-9-]+\b')
]

ERROR_DRIVEN_TEXT_PATTERNS = [
    (r'(?i)^\s*(?:the contact|consumer|owner|driver|complainant)\s+(?:states?|stated|owns|owned|leased)\b[\s,:;-]*', ''),
    (r'(?i)(?:https?://\S+|www\.\S+|[a-z0-9._%+-]+@[a-z0-9.-]+\.[a-z]{2,})', ' contact_token '),
    (r'(?i)(?:\+?1[-.\s]*)?(?:\(?\d{3}\)?[-.\s]*)\d{3}[-.\s]*\d{4}', ' phone_token '),
    (r'(?i)\b[a-hj-npr-z0-9]{17}\b', ' vin_token '),
    (r'(?i)\b(?:case|claim|ticket|reference|tracking|repair order|work order|invoice)\s*(?:number|no|#)?\s*[:#-]?\s*[a-z0-9-]{5,}\b', ' case_id_token '),
    (r'(?i)\b(?=[a-z0-9-]{8,}\b)(?=[a-z0-9-]*[a-z])(?=[a-z0-9-]*\d)[a-z0-9-]+\b', ' id_token ')
]

CANDIDATE_SELECTION_COLS = [
    'model',
    'recall_top_1pct',
    'recall_top_2pct',
    'recall_top_5pct',
    'precision_top_5pct',
    'recall_top_10pct',
    'pr_auc',
    'brier_score',
    'lift_top_5pct'
]


def format_display_table(df, rename_map=None, cols=None, max_rows=None):
    table = df.to_frame().T.copy() if isinstance(df, pd.Series) else df.copy()
    if cols is not None:
        table = table[[col for col in cols if col in table.columns]]
    if max_rows is not None:
        table = table.head(max_rows)
    if rename_map:
        table = table.rename(columns=rename_map)
    numeric_cols = table.select_dtypes(include='number').columns
    if len(numeric_cols):
        table[numeric_cols] = table[numeric_cols].round(4)
    return table


def compact_candidate_view(selection_df, max_rows=None):
    return format_display_table(
        selection_df,
        rename_map={
            'recall_top_1pct': 'r@1%',
            'recall_top_2pct': 'r@2%',
            'recall_top_5pct': 'r@5%',
            'precision_top_5pct': 'p@5%',
            'recall_top_10pct': 'r@10%',
            'brier_score': 'brier',
            'lift_top_5pct': 'lift@5%'
        },
        cols=CANDIDATE_SELECTION_COLS,
        max_rows=max_rows
    )


def compact_budget_view(budget_df, include_model=False, max_rows=None):
    cols = ['model', 'budget_label', 'flagged_rows', 'flagged_share', 'severe_cases_captured', 'recall_within_flagged_set', 'precision_within_flagged_set', 'lift_vs_base_rate', 'score_cutoff']
    if not include_model:
        cols = cols[1:]
    return format_display_table(
        budget_df,
        rename_map={
            'budget_label': 'budget',
            'flagged_rows': 'flagged',
            'flagged_share': 'share',
            'severe_cases_captured': 'captured',
            'recall_within_flagged_set': 'recall',
            'precision_within_flagged_set': 'precision',
            'lift_vs_base_rate': 'lift',
            'score_cutoff': 'cutoff'
        },
        cols=cols,
        max_rows=max_rows
    )


def compact_result_view(result_df):
    return format_display_table(
        result_df,
        rename_map={
            'positive_rate': 'base_rate',
            'brier_score': 'brier',
            'recall_top_1pct': 'r@1%',
            'recall_top_2pct': 'r@2%',
            'recall_top_5pct': 'r@5%',
            'precision_top_5pct': 'p@5%',
            'lift_top_5pct': 'lift@5%',
            'recall_top_10pct': 'r@10%',
            'precision_top_10pct': 'p@10%',
            'lift_top_10pct': 'lift@10%'
        },
        cols=[
            'model',
            'split',
            'positive_rate',
            'pr_auc',
            'roc_auc',
            'f1',
            'precision',
            'recall',
            'brier_score',
            'recall_top_1pct',
            'recall_top_2pct',
            'recall_top_5pct',
            'precision_top_5pct',
            'lift_top_5pct',
            'recall_top_10pct',
            'precision_top_10pct',
            'lift_top_10pct'
        ]
    )


def register_state(state):
    state_registry[state['name']] = state
    return state


def get_state(name):
    return state_registry[name]


def resolve_stop_words(stop_words_label):
    if stop_words_label == 'english':
        return 'english'
    if stop_words_label == 'none':
        return None
    if stop_words_label == 'custom_keep_negation':
        return CUSTOM_STOP_WORDS
    raise ValueError(f"Unknown stop_words_label: {stop_words_label}")


def get_model_margin(model, X):
    if hasattr(model, 'decision_function'):
        return np.asarray(model.decision_function(X))
    if hasattr(model, 'predict_proba'):
        proba = np.clip(model.predict_proba(X)[:, 1], 1e-8, 1 - 1e-8)
        return np.log(proba / (1 - proba))
    raise AttributeError('Model must define decision_function or predict_proba')


def sigmoid_score(raw_score):
    return 1 / (1 + np.exp(-np.asarray(raw_score)))

def build_text_series(source_df, clean_mode='base', domain_cleanup=False, error_cleanup=False):
    text = source_df['cdescr_model_text'].fillna('').astype(str)
    text = text.str.replace(r'\s+', ' ', regex=True).str.strip()

    if clean_mode not in {'base', 'light'}:
        raise ValueError(f"clean_mode must be 'base' or 'light', got {clean_mode}")

    if clean_mode == 'light':
        text = text.str.replace(r'(?i)\[\s*x+\s*\]', ' ', regex=True)
        text = text.str.replace(
            r'(?i)information redacted pursuant to the freedom of information act \(foia\) 5 u\.s\.c\. 552\(b\)\(6\) and 49 c\.f\.r\. 512\.8',
            ' ',
            regex=True
        )
        text = text.str.replace(
            r'(?i)^\s*the contact (owns|owned|stated|leased)\\b[\s,:;-]*',
            '',
            regex=True
        )

    if error_cleanup:
        for pattern, replacement in ERROR_DRIVEN_TEXT_PATTERNS:
            text = text.str.replace(pattern, replacement, regex=True)

    if domain_cleanup:
        for pattern, replacement in DOMAIN_TEXT_PATTERNS:
            text = text.str.replace(pattern, replacement, regex=True)

    return text.str.replace(r'\s+', ' ', regex=True).str.strip()


def build_text_triplet(clean_mode='base', domain_cleanup=False, error_cleanup=False):
    return (
        build_text_series(train_df, clean_mode=clean_mode, domain_cleanup=domain_cleanup, error_cleanup=error_cleanup),
        build_text_series(valid_df, clean_mode=clean_mode, domain_cleanup=domain_cleanup, error_cleanup=error_cleanup),
        build_text_series(holdout_df, clean_mode=clean_mode, domain_cleanup=domain_cleanup, error_cleanup=error_cleanup)
    )


def build_text_preprocess(
    stop_words,
    word_max_features=DEFAULT_WORD_MAX_FEATURES,
    char_max_features=DEFAULT_CHAR_MAX_FEATURES
):
    return FeatureUnion([
        ('word_tfidf', TfidfVectorizer(
            analyzer='word',
            ngram_range=WORD_NGRAM_RANGE,
            min_df=TEXT_MIN_DF,
            max_df=0.995,
            max_features=word_max_features,
            sublinear_tf=True,
            stop_words=stop_words,
            strip_accents='unicode',
            lowercase=True,
            dtype=np.float32
        )),
        ('char_tfidf', TfidfVectorizer(
            analyzer='char_wb',
            ngram_range=CHAR_NGRAM_RANGE,
            min_df=TEXT_MIN_DF,
            max_features=char_max_features,
            sublinear_tf=True,
            strip_accents='unicode',
            lowercase=True,
            dtype=np.float32
        ))
    ])


def build_structured_matrices(train_source, valid_source, holdout_source, cat_features, num_features):
    preprocess = ColumnTransformer(
        transformers=[
            ('cat', Pipeline([
                ('impute', SimpleImputer(strategy='constant', fill_value='missing')),
                ('onehot', OneHotEncoder(handle_unknown='ignore', min_frequency=50))
            ]), cat_features),
            ('num', Pipeline([
                ('impute', SimpleImputer(strategy='median')),
                ('scale', StandardScaler())
            ]), num_features)
        ],
        remainder='drop'
    )

    def prepare(source_df):
        X = source_df[cat_features + num_features].copy()
        for col in cat_features:
            X[col] = X[col].astype('string').fillna('missing').astype(str)
        for col in num_features:
            X[col] = pd.to_numeric(X[col], errors='coerce').astype('float64')
        return X

    X_train_matrix = preprocess.fit_transform(prepare(train_source))
    X_valid_matrix = preprocess.transform(prepare(valid_source))
    X_holdout_matrix = preprocess.transform(prepare(holdout_source))
    return X_train_matrix, X_valid_matrix, X_holdout_matrix


def parse_component_group_tokens(value):
    if pd.isna(value):
        return []
    return sorted({token.strip() for token in str(value).split('|') if token and token.strip()})


def component_group_feature_name(group_name):
    slug = re.sub(r'[^a-z0-9]+', '_', str(group_name).lower()).strip('_')
    return f'cg_{slug}' if slug else 'cg_unknown'


def build_component_group_feature_frames(
    train_source,
    valid_source,
    holdout_source,
    component_case_df,
    min_train_cases=COMPONENT_GROUP_MIN_TRAIN_CASES,
    top_n=COMPONENT_GROUP_TOP_N
):
    component_feature_df = component_case_df[['odino', 'component_groups', 'component_group_count']].copy()
    component_feature_df['component_group_tokens'] = component_feature_df['component_groups'].apply(parse_component_group_tokens)

    train_component = train_source[['odino', TARGET_COL]].merge(
        component_feature_df[['odino', 'component_group_tokens']],
        on='odino',
        how='left'
    )
    exploded_train_component = train_component.explode('component_group_tokens').dropna(subset=['component_group_tokens']).copy()

    if exploded_train_component.empty:
        selected_group_summary = pd.DataFrame(columns=['component_group', 'train_case_count', 'train_primary_rate', 'feature_name'])
        selected_group_lookup = {}
    else:
        selected_group_summary = exploded_train_component.groupby('component_group_tokens', observed=False).agg(
            train_case_count=(TARGET_COL, 'size'),
            train_primary_rate=(TARGET_COL, 'mean')
        ).reset_index().rename(columns={'component_group_tokens': 'component_group'})
        selected_group_summary = (
            selected_group_summary.loc[selected_group_summary['train_case_count'] >= min_train_cases]
            .sort_values(['train_primary_rate', 'train_case_count', 'component_group'], ascending=[False, False, True])
            .head(top_n)
            .reset_index(drop=True)
        )
        selected_group_summary['feature_name'] = selected_group_summary['component_group'].apply(component_group_feature_name)
        selected_group_lookup = dict(zip(selected_group_summary['component_group'], selected_group_summary['feature_name']))

    component_feature_cols = ['component_group_count'] + selected_group_summary['feature_name'].tolist()

    def add_component_group_features(source_df):
        merged = source_df.merge(
            component_feature_df[['odino', 'component_group_count', 'component_group_tokens']],
            on='odino',
            how='left'
        )
        merged['component_group_tokens'] = merged['component_group_tokens'].apply(
            lambda values: values if isinstance(values, list) else []
        )
        merged['component_group_count'] = (
            pd.to_numeric(merged['component_group_count'], errors='coerce')
            .fillna(0)
            .astype('int64')
        )
        for group_name, feature_name in selected_group_lookup.items():
            merged[feature_name] = merged['component_group_tokens'].apply(
                lambda values: int(group_name in values)
            )
        return merged.drop(columns=['component_group_tokens'])

    train_with_component = add_component_group_features(train_source)
    valid_with_component = add_component_group_features(valid_source)
    holdout_with_component = add_component_group_features(holdout_source)

    coverage_summary = pd.DataFrame([
        {
            'split': split_name,
            'rows': len(frame),
            'component_info_share': round(float(frame['component_group_count'].gt(0).mean()), 4),
            'avg_component_group_count_when_present': round(
                float(frame.loc[frame['component_group_count'].gt(0), 'component_group_count'].mean()),
                3
            ) if frame['component_group_count'].gt(0).any() else 0.0
        }
        for split_name, frame in [
            ('train_2020_2024', train_with_component),
            ('valid_2025', valid_with_component),
            ('holdout_2026', holdout_with_component)
        ]
    ])

    return (
        train_with_component,
        valid_with_component,
        holdout_with_component,
        component_feature_cols,
        coverage_summary,
        selected_group_summary
    )

# Builds one reviewer budget summary row for a score vector
def build_review_budget_row(y_true, score, fraction):
    y_true = np.asarray(y_true).astype(bool)
    score = np.asarray(score)
    n_rows = len(score)
    n_flagged = max(1, int(np.ceil(n_rows * fraction)))
    top_idx = np.argsort(score)[::-1][:n_flagged]
    severe_captured = int(y_true[top_idx].sum())
    total_positive = int(y_true.sum())
    base_rate = float(y_true.mean()) if n_rows else np.nan
    precision = severe_captured / n_flagged if n_flagged else np.nan
    recall = severe_captured / total_positive if total_positive else np.nan
    cutoff = float(score[top_idx].min()) if n_flagged else np.nan
    lift = precision / base_rate if base_rate else np.nan
    return {
        'budget_fraction': fraction,
        'budget_label': REVIEW_BUDGET_LABELS[fraction],
        'flagged_rows': n_flagged,
        'flagged_share': n_flagged / n_rows if n_rows else np.nan,
        'severe_cases_captured': severe_captured,
        'recall_within_flagged_set': recall,
        'precision_within_flagged_set': precision,
        'lift_vs_base_rate': lift,
        'score_cutoff': cutoff
    }


# Builds the operating point table for one model and split
def build_operating_point_table(model_name, split_name, y_true, score, budgets=None):
    budgets = REVIEW_BUDGETS if budgets is None else budgets
    rows = []
    for fraction in budgets:
        row = build_review_budget_row(y_true, score, fraction)
        row['model'] = model_name
        row['split'] = split_name
        rows.append(row)
    return pd.DataFrame(rows)


# Assigns urgency bands by rank so the review buckets stay stable even when scores tie
def assign_urgency_bands(score):
    score_series = pd.Series(score)
    rank_share = score_series.rank(method='first', ascending=False, pct=True)
    band_values = np.select(
        [rank_share <= 0.01, rank_share <= 0.05, rank_share <= 0.10],
        ['high_urgency', 'priority_review', 'watch'],
        default='routine'
    )
    return pd.Series(band_values, index=score_series.index)


def build_valid_review_frame(score):
    review_df = valid_df.copy()
    suspicious_flag_cols = [col for col in SUSPICIOUS_FLAG_CANDIDATES if col in review_df.columns]
    if suspicious_flag_cols:
        review_df[suspicious_flag_cols] = review_df[suspicious_flag_cols].fillna(False).astype(bool)
    review_df['suspicious_value_any'] = review_df[suspicious_flag_cols].any(axis=1) if suspicious_flag_cols else False
    review_df['urgency_score'] = score
    review_df['urgency_band'] = pd.Categorical(
        assign_urgency_bands(review_df['urgency_score']),
        categories=URGENCY_BAND_ORDER,
        ordered=True
    )
    review_df['review_month'] = review_df['ldate'].dt.to_period('M').astype(str)
    review_cols = BASE_REVIEW_COLS[:-1] + suspicious_flag_cols + BASE_REVIEW_COLS[-1:]
    return review_df, suspicious_flag_cols, review_cols


def summarize_review_frame(review_df):
    band_summary = review_df['urgency_band'].value_counts(sort=False).rename_axis('urgency_band').reset_index(name='rows')
    band_summary['share'] = band_summary['rows'] / len(review_df)
    monthly_review_burden = review_df.groupby(['review_month', 'urgency_band'], observed=False).size().unstack(fill_value=0).reset_index()
    return band_summary, monthly_review_burden


# Chooses the urgency rule using the notebook decision policy
def choose_urgency_model(selection_df, recall_tolerance=0.01):
    selection_df = selection_df.copy()

    best_recall = selection_df['recall_top_5pct'].max()
    recall_pool = selection_df[selection_df['recall_top_5pct'] >= best_recall - recall_tolerance].copy()

    best_precision = recall_pool['precision_top_5pct'].max()
    precision_pool = recall_pool[np.isclose(recall_pool['precision_top_5pct'], best_precision, atol=1e-6)]

    best_pr_auc = precision_pool['pr_auc'].max()
    pr_pool = precision_pool[np.isclose(precision_pool['pr_auc'], best_pr_auc, atol=1e-6)].copy()

    best_brier = pr_pool['brier_score'].min()
    winner = pr_pool[np.isclose(pr_pool['brier_score'], best_brier, atol=1e-6)].sort_values('model').iloc[0]

    selection_df = selection_df.sort_values(
        ['recall_top_5pct', 'precision_top_5pct', 'pr_auc', 'brier_score', 'model'],
        ascending=[False, False, False, True, True]
    ).reset_index(drop=True)
    return selection_df, winner

# Builds a linear model with stochastic gradient descent and early stopping for fast iteration
def build_linear_sgd_model(alpha=DEFAULT_ALPHA):
    return SGDClassifier(
        loss='log_loss',
        penalty='l2',
        alpha=alpha,
        max_iter=100,
        tol=1e-3,
        class_weight='balanced',
        random_state=RANDOM_SEED,
        early_stopping=True,
        validation_fraction=0.05,
        n_iter_no_change=5
    )


# Keeps one row per model and split so rerunning a notebook cell refreshes the result instead of duplicating it
def upsert_result(row):
    for idx, existing_row in enumerate(severity_results):
        if existing_row['model'] == row['model'] and existing_row['split'] == row['split']:
            severity_results[idx] = row
            return row
    severity_results.append(row)
    return row


# Evaluates the full set of metrics for a given model and dataset, then updates the severity_results list
def build_score_row(model_name, split_name, y_true, score, threshold=0.5):
    pred = score >= threshold
    row = {
        'model': model_name,
        'split': split_name,
        'rows': len(y_true),
        'positive_rate': float(np.mean(y_true)),
        'pr_auc': float(average_precision_score(y_true, score)),
        'roc_auc': float(roc_auc_score(y_true, score)),
        'f1': float(f1_score(y_true, pred, zero_division=0)),
        'precision': float(precision_score(y_true, pred, zero_division=0)),
        'recall': float(recall_score(y_true, pred, zero_division=0)),
        'brier_score': float(brier_score_loss(y_true, score))
    }
    for fraction, label in REVIEW_BUDGET_LABELS.items():
        budget_row = build_review_budget_row(y_true, score, fraction)
        row[f'recall_{label}'] = float(budget_row['recall_within_flagged_set'])
        if fraction in {0.05, 0.10}:
            row[f'precision_{label}'] = float(budget_row['precision_within_flagged_set'])
            row[f'lift_{label}'] = float(budget_row['lift_vs_base_rate'])
    return row


def evaluate_scores(model_name, split_name, y_true, score, threshold=0.5):
    row = build_score_row(model_name, split_name, y_true, score, threshold=threshold)
    return upsert_result(row)


# Displays the current severity_results as a sorted DataFrame
def show_results(display_table=True):
    results_df = pd.DataFrame(severity_results)
    if results_df.empty:
        if display_table:
            display(results_df)
        return results_df
    results_df = results_df.sort_values(['split', 'pr_auc', 'model'], ascending=[True, False, True]).reset_index(drop=True)
    if display_table:
        display(results_df)
    return results_df


def get_result_row(model_name, split_name='valid_2025'):
    results_df = show_results(display_table=False)
    row = results_df.loc[
        results_df['model'].eq(model_name) & results_df['split'].eq(split_name)
    ]
    if row.empty:
        raise KeyError(f"Missing result row for {model_name} on {split_name}")
    return row.iloc[0].copy()


def build_candidate_selection_table(model_names, split_name='valid_2025'):
    results_df = show_results(display_table=False)
    selection_df = results_df.loc[
        results_df['split'].eq(split_name) & results_df['model'].isin(model_names),
        CANDIDATE_SELECTION_COLS
    ].copy()
    if selection_df.empty:
        raise ValueError(f"No rows found for models: {model_names}")
    return selection_df.sort_values(
        ['recall_top_5pct', 'precision_top_5pct', 'pr_auc', 'brier_score', 'model'],
        ascending=[False, False, False, True, True]
    ).reset_index(drop=True)


def choose_between_candidates(model_names, current_name=None, split_name='valid_2025'):
    selection_df = build_candidate_selection_table(model_names, split_name=split_name)
    selection_df, winner = choose_urgency_model(selection_df)
    if current_name is None or current_name not in selection_df['model'].values:
        return selection_df, winner

    current_row = selection_df.loc[selection_df['model'].eq(current_name)].iloc[0]
    current_matches_winner = (
        abs(float(current_row['recall_top_5pct']) - float(winner['recall_top_5pct'])) <= 0.01
        and np.isclose(current_row['precision_top_5pct'], winner['precision_top_5pct'], atol=1e-4)
        and np.isclose(current_row['pr_auc'], winner['pr_auc'], atol=1e-4)
        and np.isclose(current_row['brier_score'], winner['brier_score'], atol=1e-4)
    )
    return selection_df, current_row if current_matches_winner else winner


def fit_linear_candidate_from_matrices(
    model_name,
    X_train,
    X_valid,
    X_holdout,
    y_train,
    y_valid,
    y_holdout,
    alpha=DEFAULT_ALPHA,
    architecture='linear',
    settings=None
):
    model = build_linear_sgd_model(alpha=alpha)
    model.fit(X_train, y_train)

    valid_margin = get_model_margin(model, X_valid)
    holdout_margin = get_model_margin(model, X_holdout)
    valid_score = sigmoid_score(valid_margin)
    holdout_score = sigmoid_score(holdout_margin)

    evaluate_scores(model_name, 'valid_2025', y_valid, valid_score)

    state = {
        'name': model_name,
        'architecture': architecture,
        'model': model,
        'valid_score': valid_score,
        'holdout_score': holdout_score,
        'valid_calibration_input': valid_margin,
        'holdout_calibration_input': holdout_margin,
        'settings': settings or {}
    }
    return register_state(state)


def build_text_matrices(
    train_text,
    valid_text,
    holdout_text,
    stop_words_label='english',
    word_max_features=DEFAULT_WORD_MAX_FEATURES,
    char_max_features=DEFAULT_CHAR_MAX_FEATURES
):
    preprocess = build_text_preprocess(
        resolve_stop_words(stop_words_label),
        word_max_features=word_max_features,
        char_max_features=char_max_features
    )
    return (
        preprocess.fit_transform(train_text),
        preprocess.transform(valid_text),
        preprocess.transform(holdout_text)
    )


def fit_text_candidate(
    model_name,
    train_text,
    valid_text,
    holdout_text,
    y_train,
    y_valid,
    y_holdout,
    stop_words_label='english',
    clean_mode='base',
    word_max_features=DEFAULT_WORD_MAX_FEATURES,
    char_max_features=DEFAULT_CHAR_MAX_FEATURES,
    domain_cleanup=False,
    error_cleanup=False,
    alpha=DEFAULT_ALPHA,
    settings=None
):
    X_train, X_valid, X_holdout = build_text_matrices(
        train_text,
        valid_text,
        holdout_text,
        stop_words_label=stop_words_label,
        word_max_features=word_max_features,
        char_max_features=char_max_features
    )

    candidate_settings = {
        'stop_words_label': stop_words_label,
        'clean_mode': clean_mode,
        'word_max_features': word_max_features,
        'char_max_features': char_max_features,
        'domain_cleanup': domain_cleanup,
        'error_cleanup': error_cleanup,
        'alpha': alpha,
        **(settings or {})
    }

    return fit_linear_candidate_from_matrices(
        model_name,
        X_train,
        X_valid,
        X_holdout,
        y_train,
        y_valid,
        y_holdout,
        alpha=alpha,
        architecture='text',
        settings=candidate_settings
    )


def fit_early_fusion_candidate(
    model_name,
    X_train_structured,
    X_valid_structured,
    X_holdout_structured,
    train_text,
    valid_text,
    holdout_text,
    y_train,
    y_valid,
    y_holdout,
    stop_words_label='english',
    clean_mode='base',
    word_max_features=DEFAULT_WORD_MAX_FEATURES,
    char_max_features=DEFAULT_CHAR_MAX_FEATURES,
    domain_cleanup=False,
    error_cleanup=False,
    alpha=DEFAULT_ALPHA,
    settings=None
):
    X_train_text, X_valid_text, X_holdout_text = build_text_matrices(
        train_text,
        valid_text,
        holdout_text,
        stop_words_label=stop_words_label,
        word_max_features=word_max_features,
        char_max_features=char_max_features
    )

    X_train = sparse.hstack([X_train_structured, X_train_text], format='csr')
    X_valid = sparse.hstack([X_valid_structured, X_valid_text], format='csr')
    X_holdout = sparse.hstack([X_holdout_structured, X_holdout_text], format='csr')

    candidate_settings = {
        'stop_words_label': stop_words_label,
        'clean_mode': clean_mode,
        'word_max_features': word_max_features,
        'char_max_features': char_max_features,
        'domain_cleanup': domain_cleanup,
        'error_cleanup': error_cleanup,
        'alpha': alpha,
        **(settings or {})
    }

    return fit_linear_candidate_from_matrices(
        model_name,
        X_train,
        X_valid,
        X_holdout,
        y_train,
        y_valid,
        y_holdout,
        alpha=alpha,
        architecture='early_fusion',
        settings=candidate_settings
    )


def build_late_fusion_state(model_name, structured_state, text_state, weight):
    valid_score = (weight * text_state['valid_score']) + ((1 - weight) * structured_state['valid_score'])
    holdout_score = (weight * text_state['holdout_score']) + ((1 - weight) * structured_state['holdout_score'])

    evaluate_scores(model_name, 'valid_2025', y_valid, valid_score)

    state = {
        'name': model_name,
        'architecture': 'late_fusion',
        'model': None,
        'valid_score': valid_score,
        'holdout_score': holdout_score,
        'valid_calibration_input': valid_score,
        'holdout_calibration_input': holdout_score,
        'settings': {
            'weight': weight,
            'text_component_name': text_state['name'],
            'structured_component_name': structured_state['name'],
            'uses_cohort_history': bool(structured_state['settings'].get('uses_cohort_history', False)),
            'uses_component_groups': bool(structured_state['settings'].get('uses_component_groups', False))
        }
    }
    return register_state(state)


def get_text_settings_from_state(state):
    text_state = get_state(state['settings']['text_component_name']) if state['architecture'] == 'late_fusion' else state
    return {
        'stop_words_label': text_state['settings']['stop_words_label'],
        'clean_mode': text_state['settings']['clean_mode'],
        'word_max_features': text_state['settings']['word_max_features'],
        'char_max_features': text_state['settings']['char_max_features'],
        'domain_cleanup': text_state['settings'].get('domain_cleanup', False),
        'error_cleanup': text_state['settings'].get('error_cleanup', False),
        'alpha': text_state['settings']['alpha']
    }

def get_current_text_state(hybrid_state):
    return get_state(hybrid_state['settings']['text_component_name']) if hybrid_state['architecture'] == 'late_fusion' else base_text_state


def fit_text_hybrid_variant(config, text_settings, structured_state, structured_matrices, hybrid_state, feature_block='core_structured'):
    clean_mode = config.get('clean_mode', text_settings['clean_mode'])
    domain_cleanup = config.get('domain_cleanup', text_settings.get('domain_cleanup', False))
    error_cleanup = config.get('error_cleanup', text_settings.get('error_cleanup', False))
    train_text, valid_text, holdout_text = build_text_triplet(
        clean_mode=clean_mode,
        domain_cleanup=domain_cleanup,
        error_cleanup=error_cleanup
    )

    text_state = fit_text_candidate(
        f"text_{config['name']}",
        train_text,
        valid_text,
        holdout_text,
        y_train,
        y_valid,
        y_holdout,
        stop_words_label=text_settings['stop_words_label'],
        clean_mode=clean_mode,
        word_max_features=text_settings['word_max_features'],
        char_max_features=text_settings['char_max_features'],
        domain_cleanup=domain_cleanup,
        error_cleanup=error_cleanup,
        alpha=text_settings['alpha']
    )

    if hybrid_state['architecture'] == 'late_fusion':
        hybrid_variant = build_late_fusion_state(
            f"hybrid_late_fusion_{config['name']}",
            structured_state,
            text_state,
            hybrid_state['settings']['weight']
        )
    else:
        hybrid_variant = fit_early_fusion_candidate(
            f"hybrid_early_fusion_{config['name']}",
            structured_matrices[0],
            structured_matrices[1],
            structured_matrices[2],
            train_text,
            valid_text,
            holdout_text,
            y_train,
            y_valid,
            y_holdout,
            stop_words_label=text_settings['stop_words_label'],
            clean_mode=clean_mode,
            word_max_features=text_settings['word_max_features'],
            char_max_features=text_settings['char_max_features'],
            domain_cleanup=domain_cleanup,
            error_cleanup=error_cleanup,
            alpha=text_settings['alpha'],
            settings={
                'feature_block': feature_block,
                'uses_cohort_history': False,
                'uses_component_groups': bool(structured_state['settings'].get('uses_component_groups', False))
            }
        )
    return text_state, hybrid_variant


def run_text_hybrid_variant_search(
    label,
    variant_configs,
    text_settings,
    structured_state,
    structured_matrices,
    hybrid_state,
    feature_block='core_structured'
):
    text_states = []
    hybrid_states = []
    current_text_state = get_current_text_state(hybrid_state)

    for config in variant_configs:
        text_state, hybrid_variant = fit_text_hybrid_variant(
            config,
            text_settings,
            structured_state,
            structured_matrices,
            hybrid_state,
            feature_block=feature_block
        )
        text_states.append(text_state)
        hybrid_states.append(hybrid_variant)

    text_selection = build_candidate_selection_table(
        [current_text_state['name']] + [state['name'] for state in text_states]
    )
    hybrid_selection, hybrid_winner = choose_between_candidates(
        [hybrid_state['name']] + [state['name'] for state in hybrid_states],
        current_name=hybrid_state['name']
    )

    print(f"{label}: text-only")
    display(compact_candidate_view(text_selection))

    print(f"{label}: hybrid")
    display(compact_candidate_view(hybrid_selection))

    return update_promoted_state(
        hybrid_state,
        hybrid_winner,
        {state['name']: state for state in hybrid_states},
        label
    )


def get_structured_matrix_bundle(uses_component_groups=False):
    return component_structured_matrix_bundle if uses_component_groups else structured_matrix_bundle


def update_promoted_state(current_state, winner, state_lookup, label):
    if winner['model'] == current_state['name']:
        print(f"{label} kept the current promoted hybrid:", current_state['name'])
        return current_state
    promoted_state = dict(state_lookup[winner['model']])
    print(f"Promoted after {label}:", promoted_state['name'])
    return promoted_state

## 6. Naive Baseline

A baseline is the simplest model we must beat. If a more complex model cannot beat this, it is not worth using. This baseline gives every complaint the same score based only on how common severe complaints are in the training data.

In [80]:
dummy_model = DummyClassifier(strategy='prior', random_state=RANDOM_SEED)
dummy_model.fit(train_df[['odino']], y_train)

dummy_valid_score = dummy_model.predict_proba(valid_df[['odino']])[:, 1]
dummy_holdout_score = dummy_model.predict_proba(holdout_df[['odino']])[:, 1]

evaluate_scores('dummy_prior', 'valid_2025', y_valid, dummy_valid_score)
dummy_state = register_state({
    'name': 'dummy_prior',
    'architecture': 'baseline',
    'model': dummy_model,
    'valid_score': dummy_valid_score,
    'holdout_score': dummy_holdout_score,
    'valid_calibration_input': dummy_valid_score,
    'holdout_calibration_input': dummy_holdout_score,
    'settings': {'baseline': 'class_prior'}
})

display(build_candidate_selection_table(['dummy_prior']))

,model,recall_top_1pct,recall_top_2pct,recall_top_5pct,precision_top_5pct,recall_top_10pct,pr_auc,brier_score,lift_top_5pct
0,dummy_prior,0.012085,0.022114,0.05837,0.061351,0.119054,0.052565,0.049898,1.167159


## 7. Structured Only Baseline

This model uses only the non-text columns. It keeps a conservative set of complaint metadata and numeric fields and avoids very sparse decode columns that mostly act like data availability signals.

In [81]:
# Derive complaint timing fields up front so the structured core can use them safely
for frame in [train_df, valid_df, holdout_df]:
    frame['complaint_year'] = frame['ldate'].dt.year.astype('int64')
    frame['complaint_month'] = frame['ldate'].dt.month.astype('int64')

# Leakage-safe structured core from the cleaning policy
structured_cat_features = [
    'mfr_name',
    'maketxt',
    'modeltxt',
    'state',
    'cmpl_type',
    'drive_train',
    'fuel_type',
    'police_rpt_yn',
    'repaired_yn',
    'orig_owner_yn'
]

structured_num_features = [
    'yeartxt',
    'miles',
    'veh_speed',
    'lag_days_safe',
    'miles_missing_flag',
    'veh_speed_missing_flag',
    'faildate_trusted_flag',
    'faildate_untrusted_flag',
    'component_count',
    'row_count',
    'complaint_year',
    'complaint_month'
]

structured_cat_features = [col for col in structured_cat_features if col in df.columns]
structured_num_features = [col for col in structured_num_features if col in df.columns]

structured_matrix_bundle = build_structured_matrices(
    train_df,
    valid_df,
    holdout_df,
    structured_cat_features,
    structured_num_features
)
X_train_structured_matrix, X_valid_structured_matrix, X_holdout_structured_matrix = structured_matrix_bundle

base_structured_state = fit_linear_candidate_from_matrices(
    'structured_core_sgd',
    X_train_structured_matrix,
    X_valid_structured_matrix,
    X_holdout_structured_matrix,
    y_train,
    y_valid,
    y_holdout,
    alpha=DEFAULT_ALPHA,
    architecture='structured',
    settings={
        'feature_block': 'core_structured',
        'uses_cohort_history': False,
        'uses_component_groups': False,
        'cat_features': structured_cat_features,
        'num_features': structured_num_features
    }
)

display(build_candidate_selection_table(['structured_core_sgd']))

,model,recall_top_1pct,recall_top_2pct,recall_top_5pct,precision_top_5pct,recall_top_10pct,pr_auc,brier_score,lift_top_5pct
0,structured_core_sgd,0.173052,0.342247,0.429416,0.451351,0.477244,0.444691,0.075919,8.586585


## 8. Text Only Baseline

This model uses only the complaint narrative. It applies TF-IDF to turns word and short character patterns into numeric text features. I intentially kept settings moderate so the notebook remains runnable on local hardware.

In [82]:
train_text_base, valid_text_base, holdout_text_base = build_text_triplet(clean_mode='base')

base_text_state = fit_text_candidate(
    'text_tfidf_word_char_sgd',
    train_text_base,
    valid_text_base,
    holdout_text_base,
    y_train,
    y_valid,
    y_holdout,
    stop_words_label='english',
    clean_mode='base',
    word_max_features=DEFAULT_WORD_MAX_FEATURES,
    char_max_features=DEFAULT_CHAR_MAX_FEATURES,
    alpha=DEFAULT_ALPHA
)

display(build_candidate_selection_table(['text_tfidf_word_char_sgd']))


,model,recall_top_1pct,recall_top_2pct,recall_top_5pct,precision_top_5pct,recall_top_10pct,pr_auc,brier_score,lift_top_5pct
0,text_tfidf_word_char_sgd,0.18848,0.372332,0.742864,0.780811,0.870661,0.822153,0.043138,14.854278


## 9. First Hybrid Model

This first hybrid model combines the structured features and the text features in one model. It lets us check whether the non-text fields add anything once the complaint narrative is already available.


In [83]:
base_hybrid_state = fit_early_fusion_candidate(
    'hybrid_structured_text_sgd',
    X_train_structured_matrix,
    X_valid_structured_matrix,
    X_holdout_structured_matrix,
    train_text_base,
    valid_text_base,
    holdout_text_base,
    y_train,
    y_valid,
    y_holdout,
    stop_words_label='english',
    clean_mode='base',
    word_max_features=DEFAULT_WORD_MAX_FEATURES,
    char_max_features=DEFAULT_CHAR_MAX_FEATURES,
    alpha=DEFAULT_ALPHA,
    settings={
        'feature_block': 'core_structured',
        'uses_cohort_history': False,
        'uses_component_groups': False
    }
)

base_state_lookup = {
    'dummy_prior': dummy_state,
    'structured_core_sgd': base_structured_state,
    'text_tfidf_word_char_sgd': base_text_state,
    'hybrid_structured_text_sgd': base_hybrid_state
}

display(build_candidate_selection_table(list(base_state_lookup)))

,model,recall_top_1pct,recall_top_2pct,recall_top_5pct,precision_top_5pct,recall_top_10pct,pr_auc,brier_score,lift_top_5pct
0,hybrid_structured_text_sgd,0.188737,0.372589,0.744150,0.782162,0.867061,0.817049,0.024188,14.879987
1,text_tfidf_word_char_sgd,0.188480,0.372332,0.742864,0.780811,0.870661,0.822153,0.043138,14.854278
2,structured_core_sgd,0.173052,0.342247,0.429416,0.451351,0.477244,0.444691,0.075919,8.586585
3,dummy_prior,0.012085,0.022114,0.058370,0.061351,0.119054,0.052565,0.049898,1.167159


## 10. Compare Review Queues And Pick A Starting Rule

Here we turn each model's scores into actual review queues at the 1%, 2%, 5%, and 10% levels.

The main queue measures are:
- recall: how many truly severe complaints are captured
- precision: how clean the flagged queue is
- lift: how much better the queue is than the overall severe rate in the full data

We choose the current rule simply as::
- best recall at the top 5% review budget
- if two models are within 0.01 recall at 5%, choose the one with better precision at 5%
- if still tied, choose higher PR-AUC, which is a summary measure of ranking quality for rare targets (Precision Recall Area Under Curve, like ROC AUC)
- if still tied, choose lower Brier score, which rewards scores that behave more like sensible probabilities

In [84]:
valid_budget_df = pd.concat(
    [
        build_operating_point_table(model_name, 'valid_2025', y_valid, state['valid_score'])
        for model_name, state in base_state_lookup.items()
    ],
    ignore_index=True
).sort_values(
    ['budget_fraction', 'recall_within_flagged_set', 'precision_within_flagged_set'],
    ascending=[True, False, False]
).reset_index(drop=True)

display(compact_budget_view(valid_budget_df, include_model=True))

selection_df, selected_model_row = choose_urgency_model(build_candidate_selection_table(list(base_state_lookup)))
selected_model_name = selected_model_row['model']
selected_model_state = base_state_lookup[selected_model_name]

print("\n\nSelected urgency rule:", selected_model_name)
print("Primary review budget:", "top 5%")
print("Top 5% recall:", round(float(selected_model_row['recall_top_5pct']), 4))
print("Top 5% precision:", round(float(selected_model_row['precision_top_5pct']), 4))
print("Validation PR-AUC:", round(float(selected_model_row['pr_auc']), 4))
print("Validation Brier score:", round(float(selected_model_row['brier_score']), 4))

display(compact_candidate_view(selection_df))

,model,budget,flagged,share,captured,recall,precision,lift,cutoff
0,hybrid_structured_text_sgd,top_1pct,740,0.01,734,0.1887,0.9919,18.8699,1.0000
1,text_tfidf_word_char_sgd,top_1pct,740,0.01,733,0.1885,0.9905,18.8442,0.9996
2,structured_core_sgd,top_1pct,740,0.01,673,0.1731,0.9095,17.3017,1.0000
3,dummy_prior,top_1pct,740,0.01,47,0.0121,0.0635,1.2083,0.0624
4,hybrid_structured_text_sgd,top_2pct,1480,0.02,1449,0.3726,0.9791,18.6257,1.0000
5,text_tfidf_word_char_sgd,top_2pct,1480,0.02,1448,0.3723,0.9784,18.6128,0.9952
6,structured_core_sgd,top_2pct,1480,0.02,1331,0.3422,0.8993,17.1089,1.0000
7,dummy_prior,top_2pct,1480,0.02,86,0.0221,0.0581,1.1055,0.0624
8,hybrid_structured_text_sgd,top_5pct,3700,0.05,2894,0.7442,0.7822,14.8800,0.7286
9,text_tfidf_word_char_sgd,top_5pct,3700,0.05,2889,0.7429,0.7808,14.8543,0.8012




Selected urgency rule: hybrid_structured_text_sgd
Primary review budget: top 5%
Top 5% recall: 0.7442
Top 5% precision: 0.7822
Validation PR-AUC: 0.817
Validation Brier score: 0.0242


,model,r@1%,r@2%,r@5%,p@5%,r@10%,pr_auc,brier,lift@5%
0,hybrid_structured_text_sgd,0.1887,0.3726,0.7442,0.7822,0.8671,0.8170,0.0242,14.8800
1,text_tfidf_word_char_sgd,0.1885,0.3723,0.7429,0.7808,0.8707,0.8222,0.0431,14.8543
2,structured_core_sgd,0.1731,0.3422,0.4294,0.4514,0.4772,0.4447,0.0759,8.5866
3,dummy_prior,0.0121,0.0221,0.0584,0.0614,0.1191,0.0526,0.0499,1.1672


## 11. Review The Chosen Queue

Once a starting rule is chosen, we inspect what it would look like in practice. This includes queue sizes, monthly review burden, highest risk complaints, obvious false positives, and severe complaints that still fall outside the top 10%.

In [85]:
selected_valid_df, suspicious_flag_cols, review_cols = build_valid_review_frame(selected_model_state['valid_score'])
band_summary, monthly_review_burden = summarize_review_frame(selected_valid_df)
review_n = 8

top_risk_df = selected_valid_df.sort_values('urgency_score', ascending=False)[review_cols].head(review_n)
false_positive_df = selected_valid_df.loc[
    selected_valid_df['urgency_band'].eq('high_urgency')
    & ~selected_valid_df[TARGET_COL],
    review_cols
].sort_values('urgency_score', ascending=False).head(review_n)
missed_severe_df = selected_valid_df.loc[
    selected_valid_df[TARGET_COL]
    & selected_valid_df['urgency_band'].eq('routine'),
    review_cols
].sort_values('urgency_score', ascending=False).head(review_n)

print("Urgency band counts on valid_2025")
display(band_summary)

print("Monthly review burden on valid_2025")
display(monthly_review_burden)

print(f"Top {review_n} highest-scored complaints on valid_2025")
display(top_risk_df)

print("Top-band false positives on valid_2025")
display(false_positive_df)

print("Missed severe complaints outside the top 10% on valid_2025")
display(missed_severe_df)

Urgency band counts on valid_2025


,urgency_band,rows,share
0,high_urgency,739,0.009989
1,priority_review,2960,0.040008
2,watch,3699,0.049997
3,routine,66587,0.900007


Monthly review burden on valid_2025


urgency_band,review_month,high_urgency,priority_review,watch,routine
0,2025-01,41,214,325,5904
1,2025-02,51,199,250,5092
2,2025-03,54,206,286,5626
3,2025-04,55,176,271,5570
4,2025-05,55,227,264,5471
5,2025-06,46,228,309,5627
6,2025-07,66,296,350,6521
7,2025-08,74,292,339,5793
8,2025-09,75,309,335,5640
9,2025-10,77,279,321,5476


Top 8 highest-scored complaints on valid_2025


,odino,ldate,mfr_name,maketxt,modeltxt,yeartxt,urgency_score,urgency_band,suspicious_value_any,severity_primary_flag,severity_broad_flag,flag_speed_999,flag_speed_high,flag_miles_high,flag_injured_99,flag_deaths_99,cdescr_model_text
295452,11638081,2025-01-23,"BMW OF NORTH AMERICA, LLC",MINI,COOPER,2009,1.0,high_urgency,False,True,True,False,False,False,False,False,The contact owned a 2009 Mini Cooper. The cont...
314449,11657299,2025-04-28,"TESLA, INC.",TESLA,MODEL Y,2023,1.0,high_urgency,False,True,True,False,False,False,False,False,The contact owned a 2023 Tesla Model Y. The co...
310843,11653659,2025-04-09,"GENERAL MOTORS, LLC",CHEVROLET,TAHOE,2022,1.0,high_urgency,False,True,True,False,False,False,False,False,The contact owned a 2022 Chevrolet Tahoe. The ...
352572,11695774,2025-10-27,"TESLA, INC.",TESLA,MODEL Y,2026,1.0,high_urgency,False,True,True,False,False,False,False,False,The contact owned a 2026 Tesla Model Y. The co...
347888,11691052,2025-10-02,FORD MOTOR COMPANY,FORD,BRONCO SPORT,2023,1.0,high_urgency,False,True,True,False,False,False,False,False,The contact owned a 2023 Ford Bronco Sport. Th...
311904,11654729,2025-04-15,FORD MOTOR COMPANY,FORD,ESCAPE,2016,1.0,high_urgency,False,True,True,False,False,False,False,False,The contact's wife owned a 2016 Ford Escape. T...
340139,11683199,2025-08-26,FORD MOTOR COMPANY,FORD,MUSTANG,2013,1.0,high_urgency,False,True,True,False,False,False,False,False,The contact owned a 2013 Ford Mustang. The con...
358046,11701277,2025-11-24,FORD MOTOR COMPANY,MERCURY,SABLE,2008,1.0,high_urgency,False,True,True,False,False,False,False,False,The contact owns a 2008 Mercury Sable. While t...


Top-band false positives on valid_2025


,odino,ldate,mfr_name,maketxt,modeltxt,yeartxt,urgency_score,urgency_band,suspicious_value_any,severity_primary_flag,severity_broad_flag,flag_speed_999,flag_speed_high,flag_miles_high,flag_injured_99,flag_deaths_99,cdescr_model_text
343980,11687059,2025-09-13,FORD MOTOR COMPANY,LINCOLN,MKC,2017,1.0,high_urgency,False,False,False,False,False,False,False,False,None of my airbags deployed in a one vehicle a...
311586,11654407,2025-04-14,"CHRYSLER (FCA US, LLC)",JEEP,CHEROKEE,2019,1.0,high_urgency,False,False,True,False,False,False,False,False,The contact owned a 2019 Jeep Cherokee. The co...
341430,11684498,2025-09-02,"KIA AMERICA, INC.",KIA,EV6,2025,1.0,high_urgency,False,False,False,False,False,False,False,False,I was driving over to walk my friends dog and ...
339585,11682640,2025-08-24,HYUNDAI MOTOR AMERICA,HYUNDAI,PALISADE,2020,1.0,high_urgency,False,False,False,False,False,False,False,False,"On [XXX], while driving on the freeway, my 202..."
306944,11649705,2025-03-21,MAZDA MOTOR CORP.,MAZDA,CX-30,2023,1.0,high_urgency,False,False,False,False,False,False,False,False,While driving at approximately 20 mph on a res...
338756,11681807,2025-08-20,TOYOTA MOTOR CORPORATION,TOYOTA,COROLLA,2003,1.0,high_urgency,False,False,True,False,False,False,False,False,The contact owns a 2003 Toyota Corolla. The co...


Missed severe complaints outside the top 10% on valid_2025


,odino,ldate,mfr_name,maketxt,modeltxt,yeartxt,urgency_score,urgency_band,suspicious_value_any,severity_primary_flag,severity_broad_flag,flag_speed_999,flag_speed_high,flag_miles_high,flag_injured_99,flag_deaths_99,cdescr_model_text
301993,11644692,2025-02-25,"CHRYSLER (FCA US, LLC)",JEEP,PATRIOT,2016,0.126781,routine,False,True,True,False,False,False,False,False,Sitting at a red light with foot on brake and ...
323965,11666910,2025-06-14,HONDA (AMERICAN HONDA MOTOR CO.),HONDA,ELEMENT,2007,0.125397,routine,False,True,True,False,False,False,False,False,"Rear Trailing Arm Rusted Out, Car Can not be r..."
357493,11700720,2025-11-21,"GENERAL MOTORS, LLC",GMC,YUKON,2024,0.124591,routine,False,True,True,False,False,False,False,False,Start transmission was replaced by dealer at 7...
352083,11695282,2025-10-23,"GENERAL MOTORS, LLC",GMC,SIERRA 2500,2025,0.124481,routine,False,True,True,False,False,False,False,False,The accident occurred when the tailgate opened...
305363,11648104,2025-03-13,FORD MOTOR COMPANY,FORD,BRONCO,2021,0.124311,routine,False,True,True,False,False,False,False,False,Our family was traveling from Lake George New ...
354923,11698139,2025-11-07,"BMW OF NORTH AMERICA, LLC",MINI,CLUBMAN,2016,0.123513,routine,False,True,True,False,False,False,False,False,I purchased a Septor 4xs tire from Walmart in ...
343462,11686539,2025-09-11,"GENERAL MOTORS, LLC",CHEVROLET,TRAVERSE,2024,0.122956,routine,False,True,True,False,False,False,False,False,"On July 31st, while backing up on a clear road..."
347487,11690649,2025-09-30,FORD MOTOR COMPANY,FORD,F-350,2020,0.122921,routine,False,True,True,False,False,False,False,False,I was driving when the steering column upper s...


## 12. Where Text And Hybrid Disagree

Text only and hybrid can look close on summary metrics. We want to inspect where they disagree to see whether the complaints that only hybrid model is choosing are adding useful signal.

In [86]:
comparison_budget = 0.05
comparison_n = max(1, int(np.ceil(len(valid_df) * comparison_budget)))

queue_compare_df = selected_valid_df.copy()
queue_compare_df['text_score'] = base_text_state['valid_score']
queue_compare_df['hybrid_score'] = base_hybrid_state['valid_score']
queue_compare_df['score_gap'] = queue_compare_df['hybrid_score'] - queue_compare_df['text_score']

text_top_mask = np.zeros(len(queue_compare_df), dtype=bool)
text_top_mask[np.argsort(base_text_state['valid_score'])[::-1][:comparison_n]] = True
hybrid_top_mask = np.zeros(len(queue_compare_df), dtype=bool)
hybrid_top_mask[np.argsort(base_hybrid_state['valid_score'])[::-1][:comparison_n]] = True

queue_compare_df['queue_membership'] = pd.Categorical(
    np.select(
        [
            text_top_mask & hybrid_top_mask,
            text_top_mask & ~hybrid_top_mask,
            ~text_top_mask & hybrid_top_mask
        ],
        [
            'shared_top_5pct',
            'text_only_top_5pct',
            'hybrid_only_top_5pct'
        ],
        default='outside_top_5pct'
    ),
    categories=[
        'shared_top_5pct',
        'text_only_top_5pct',
        'hybrid_only_top_5pct',
        'outside_top_5pct'
    ],
    ordered=True
)

queue_overlap_summary = (
    queue_compare_df.loc[queue_compare_df['queue_membership'].ne('outside_top_5pct')]
    .groupby('queue_membership', observed=False)
    .agg(
        rows=('odino', 'size'),
        primary_rate=(TARGET_COL, 'mean'),
        broad_rate=('severity_broad_flag', 'mean'),
        suspicious_rate=('suspicious_value_any', 'mean'),
        avg_score_gap=('score_gap', 'mean')
    )
    .reset_index()
)
queue_overlap_summary['share_of_top_5_budget'] = queue_overlap_summary['rows'] / comparison_n

queue_review_cols = [
    'odino',
    'ldate',
    'mfr_name',
    'maketxt',
    'modeltxt',
    'yeartxt',
    'text_score',
    'hybrid_score',
    'score_gap',
    'suspicious_value_any',
    'severity_primary_flag',
    'severity_broad_flag',
    'cdescr_model_text'
]

text_only_df = queue_compare_df.loc[
    queue_compare_df['queue_membership'].eq('text_only_top_5pct'),
    queue_review_cols
].sort_values('text_score', ascending=False).head(8)

hybrid_only_df = queue_compare_df.loc[
    queue_compare_df['queue_membership'].eq('hybrid_only_top_5pct'),
    queue_review_cols
].sort_values('hybrid_score', ascending=False).head(8)

print("Text vs hybrid top 5% queue overlap:", round(float((text_top_mask & hybrid_top_mask).mean() / comparison_budget), 4))
print("Rows in each top 5% queue:", comparison_n)
display(format_display_table(
    queue_overlap_summary,
    rename_map={
        'queue_membership': 'queue',
        'share_of_top_5_budget': 'share',
        'avg_score_gap': 'avg_gap'
    },
    cols=[
        'queue_membership',
        'rows',
        'share_of_top_5_budget',
        'primary_rate',
        'broad_rate',
        'suspicious_rate',
        'avg_score_gap'
    ]
))

print("Text only complaints that fall out of the hybrid top 5% queue")
display(text_only_df)

print("Hybrid only complaints that enter the hybrid top 5% queue")
display(hybrid_only_df)


Text vs hybrid top 5% queue overlap: 0.8586
Rows in each top 5% queue: 3700


,queue,rows,share,primary_rate,broad_rate,suspicious_rate,avg_gap
0,shared_top_5pct,3176,0.8584,0.8454,0.853,0.0006,0.0048
1,text_only_top_5pct,524,0.1416,0.3893,0.4065,0.0057,-0.4321
2,hybrid_only_top_5pct,524,0.1416,0.3989,0.4237,0.0000,0.4112
3,outside_top_5pct,0,0.0,<NA>,<NA>,NaN,NaN


Text only complaints that fall out of the hybrid top 5% queue


,odino,ldate,mfr_name,maketxt,modeltxt,yeartxt,text_score,hybrid_score,score_gap,suspicious_value_any,severity_primary_flag,severity_broad_flag,cdescr_model_text
303621,11646344,2025-03-04,"KIA AMERICA, INC.",KIA,NIRO EV,2024,0.999997,5.011705e-03,-0.994985,False,True,True,The contact owns a 2024 Kia Niro. The contact ...
323030,11665972,2025-06-10,TOYOTA MOTOR CORPORATION,TOYOTA,CAMRY,2024,0.999973,8.660718e-05,-0.999887,False,True,True,The contact owned a 2024 Toyota Camry. The con...
345273,11688357,2025-09-19,"GENERAL MOTORS, LLC",GMC,SIERRA 1500,2000,0.999954,3.837732e-04,-0.999570,False,True,True,The contact owns a 1995 N/A GMC Sierra 1500. T...
331612,11674618,2025-07-18,"TESLA, INC.",TESLA,MODEL Y,2024,0.999637,1.663929e-05,-0.999621,False,True,True,The contact owned a 2024 Tesla Model Y. The co...
338812,11681863,2025-08-20,"MITSUBISHI MOTORS NORTH AMERICA, INC.",MITSUBISHI,OUTLANDER SPORT,2019,0.997259,1.136461e-01,-0.883613,True,True,True,Airbags did not deploy.
345265,11688349,2025-09-19,FORD MOTOR COMPANY,FORD,F-150,2000,0.995319,1.039804e-09,-0.995319,False,True,True,The contact owns a 1991 N/A Ford F-150. The co...
345143,11688226,2025-09-18,"TESLA, INC.",TESLA,MODEL Y,2026,0.992765,5.881973e-12,-0.992765,False,True,True,Our 2026 Tesla Model Y could not stop at a tra...
363018,11706271,2025-12-20,HYUNDAI MOTOR AMERICA,GENESIS,GV70,2025,0.989495,2.532349e-09,-0.989495,False,True,True,I was heading home early in the morning on 12/...


Hybrid only complaints that enter the hybrid top 5% queue


,odino,ldate,mfr_name,maketxt,modeltxt,yeartxt,text_score,hybrid_score,score_gap,suspicious_value_any,severity_primary_flag,severity_broad_flag,cdescr_model_text
356727,11699949,2025-11-17,FORD MOTOR COMPANY,FORD,EXPEDITION,2023,0.615517,0.999990,0.384473,False,False,False,The seatbelt strap malfunctioned ( locked) and...
341136,11684202,2025-08-31,"NISSAN NORTH AMERICA, INC.",NISSAN,ROGUE,2023,0.742698,0.999989,0.257291,False,False,False,My back window of my 2023 Nissan Rogue SL Blew...
361603,11704846,2025-12-13,"JAGUAR LAND ROVER NORTH AMERICA, LLC",LAND ROVER,RANGE ROVER VELAR,2021,0.723047,0.999988,0.276941,False,True,True,Well maintained vehicle I’ve been sole owner o...
326399,11669368,2025-06-26,FORD MOTOR COMPANY,FORD,ESCAPE,2013,0.755039,0.999987,0.244948,False,True,True,While coming to a stop sign the vehicle was go...
347028,11690186,2025-09-28,"TESLA, INC.",TESLA,MODEL 3,2019,0.767127,0.999982,0.232855,False,True,True,Vehicle: 2019 Tesla Model 3 Dual Motor VIN: [X...
353390,11696598,2025-10-30,HONDA (AMERICAN HONDA MOTOR CO.),HONDA,ODYSSEY,2016,0.688428,0.999981,0.311554,False,False,False,I Have a 2016 Honda Odyssey. I have had my van...
303069,11645786,2025-03-02,HYUNDAI MOTOR AMERICA,HYUNDAI,TUCSON,2025,0.781210,0.999980,0.218770,False,True,True,I am filing a complaint regarding a serious sa...
339670,11682727,2025-08-24,FORD MOTOR COMPANY,FORD,EXPEDITION,2023,0.801049,0.999978,0.198929,False,True,True,It made a loud boom I stopped thought it was a...


## 13. Lock The Hybrid Benchmark And Try Late Fusion

At this point the early fusion hybrid is the benchmark to beat, so our goal is to now try to incrementally improve they hybrid in discrete steps. The first step will be tryin late fusion, where we keep the text and structured parts separate, then blend their scores after the fact to see whether that gives a better review queue. We'll test different weight values to see what percentage of each score is considered for the fused score.

In [87]:
promoted_hybrid_state = dict(base_hybrid_state)

print("Locked benchmark:", promoted_hybrid_state['name'])
print("Architecture:", promoted_hybrid_state['architecture'])
print("2026 policy:", "final reference only")

display(compact_candidate_view(build_candidate_selection_table([
    'text_tfidf_word_char_sgd',
    promoted_hybrid_state['name']
])))

late_fusion_weights = [0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95]
late_fusion_states = [
    build_late_fusion_state(
        f"hybrid_late_fusion_w_{weight:.2f}".replace('.', '_'),
        base_structured_state,
        base_text_state,
        weight
    )
    for weight in late_fusion_weights
]

late_fusion_selection, late_fusion_winner = choose_between_candidates(
    [promoted_hybrid_state['name']] + [state['name'] for state in late_fusion_states],
    current_name=promoted_hybrid_state['name']
)
late_fusion_lookup = {state['name']: state for state in late_fusion_states}

display(compact_candidate_view(late_fusion_selection))
promoted_hybrid_state = update_promoted_state(promoted_hybrid_state, late_fusion_winner, late_fusion_lookup, 'late-fusion search')


Locked benchmark: hybrid_structured_text_sgd
Architecture: early_fusion
2026 policy: final reference only


,model,r@1%,r@2%,r@5%,p@5%,r@10%,pr_auc,brier,lift@5%
0,hybrid_structured_text_sgd,0.1887,0.3726,0.7442,0.7822,0.8671,0.8170,0.0242,14.8800
1,text_tfidf_word_char_sgd,0.1885,0.3723,0.7429,0.7808,0.8707,0.8222,0.0431,14.8543


,model,r@1%,r@2%,r@5%,p@5%,r@10%,pr_auc,brier,lift@5%
0,hybrid_late_fusion_w_0_75,0.1895,0.3692,0.7519,0.7903,0.8827,0.8211,0.0373,15.0342
1,hybrid_late_fusion_w_0_80,0.1895,0.3692,0.7516,0.7900,0.8815,0.8240,0.0377,15.0291
2,hybrid_late_fusion_w_0_85,0.1895,0.3698,0.7508,0.7892,0.8750,0.8255,0.0385,15.0137
3,hybrid_late_fusion_w_0_70,0.1895,0.3682,0.7493,0.7876,0.8830,0.8166,0.0373,14.9828
4,hybrid_late_fusion_w_0_90,0.1895,0.3700,0.7488,0.7870,0.8740,0.8264,0.0397,14.9725
5,hybrid_late_fusion_w_0_95,0.1895,0.3710,0.7462,0.7843,0.8725,0.8262,0.0412,14.9211
6,hybrid_late_fusion_w_0_65,0.1895,0.3680,0.7457,0.7838,0.8822,0.8100,0.0376,14.9108
7,hybrid_structured_text_sgd,0.1887,0.3726,0.7442,0.7822,0.8671,0.8170,0.0242,14.8800
8,hybrid_late_fusion_w_0_60,0.1895,0.3682,0.7390,0.7768,0.8773,0.7991,0.0383,14.7772
9,hybrid_late_fusion_w_0_55,0.1895,0.3687,0.7190,0.7557,0.8663,0.7815,0.0394,14.3761


Promoted after late-fusion search: hybrid_late_fusion_w_0_75


## 14. Try Small Text Processing Changes

Next we test a small set of simple text processing choices while keeping the rest of the model the same. We want to see whether a few easy to explain text changes make the review queue better.

In [88]:
text_variant_configs = [
    {
        'name': 'english_stop_base',
        'stop_words_label': 'english',
        'clean_mode': 'base'
    },
    {
        'name': 'no_stop_base',
        'stop_words_label': 'none',
        'clean_mode': 'base'
    },
    {
        'name': 'custom_keep_negation_base',
        'stop_words_label': 'custom_keep_negation',
        'clean_mode': 'base'
    },
    {
        'name': 'custom_keep_negation_light_clean',
        'stop_words_label': 'custom_keep_negation',
        'clean_mode': 'light'
    }
]

promoted_hybrid_state = run_text_hybrid_variant_search(
    'text ablation',
    text_variant_configs,
    get_text_settings_from_state(promoted_hybrid_state),
    base_structured_state,
    structured_matrix_bundle,
    promoted_hybrid_state,
    feature_block='core_structured'
)

text ablation: text-only


,model,r@1%,r@2%,r@5%,p@5%,r@10%,pr_auc,brier,lift@5%
0,text_custom_keep_negation_light_clean,0.1885,0.3723,0.7436,0.7816,0.8712,0.8224,0.0429,14.8697
1,text_custom_keep_negation_base,0.1885,0.3723,0.7429,0.7808,0.8707,0.8222,0.0431,14.8543
2,text_english_stop_base,0.1885,0.3723,0.7429,0.7808,0.8707,0.8222,0.0431,14.8543
3,text_no_stop_base,0.1885,0.3723,0.7429,0.7808,0.8707,0.8222,0.0431,14.8543
4,text_tfidf_word_char_sgd,0.1885,0.3723,0.7429,0.7808,0.8707,0.8222,0.0431,14.8543


text ablation: hybrid


,model,r@1%,r@2%,r@5%,p@5%,r@10%,pr_auc,brier,lift@5%
0,hybrid_late_fusion_custom_keep_negation_light_...,0.1895,0.3690,0.7526,0.7911,0.8830,0.8215,0.0372,15.0497
1,hybrid_late_fusion_custom_keep_negation_base,0.1895,0.3692,0.7519,0.7903,0.8827,0.8211,0.0373,15.0342
2,hybrid_late_fusion_english_stop_base,0.1895,0.3692,0.7519,0.7903,0.8827,0.8211,0.0373,15.0342
3,hybrid_late_fusion_no_stop_base,0.1895,0.3692,0.7519,0.7903,0.8827,0.8211,0.0373,15.0342
4,hybrid_late_fusion_w_0_75,0.1895,0.3692,0.7519,0.7903,0.8827,0.8211,0.0373,15.0342


Promoted after text ablation: hybrid_late_fusion_custom_keep_negation_light_clean


## 15. Try Simple Domain Phrase Cleanup

There are a number of safety signal the reoccur throught the complaint text and normalizing them could potentially allow the text model to treat them more consistently. This may increase the severity signal of certain important phrases without changing the overall model.

ex. Phrases like "loss of power," "lost power," "loses power," and "no power" are all standardized to "no_power"

In [89]:
domain_text_variant_configs = [
    {
        'name': 'domain_core_base',
        'clean_mode': 'base',
        'domain_cleanup': True
    },
    {
        'name': 'domain_core_light',
        'clean_mode': 'light',
        'domain_cleanup': True
    }
]

promoted_hybrid_state = run_text_hybrid_variant_search(
    'domain text cleanup',
    domain_text_variant_configs,
    get_text_settings_from_state(promoted_hybrid_state),
    base_structured_state,
    structured_matrix_bundle,
    promoted_hybrid_state,
    feature_block='core_structured'
)

domain text cleanup: text-only


,model,r@1%,r@2%,r@5%,p@5%,r@10%,pr_auc,brier,lift@5%
0,text_custom_keep_negation_light_clean,0.1885,0.3723,0.7436,0.7816,0.8712,0.8224,0.0429,14.8697
1,text_domain_core_light,0.1887,0.3723,0.7431,0.7811,0.8714,0.8226,0.0428,14.8594
2,text_domain_core_base,0.1887,0.3723,0.7431,0.7811,0.8704,0.8223,0.0429,14.8594


domain text cleanup: hybrid


,model,r@1%,r@2%,r@5%,p@5%,r@10%,pr_auc,brier,lift@5%
0,hybrid_late_fusion_custom_keep_negation_light_...,0.1895,0.3690,0.7526,0.7911,0.8830,0.8215,0.0372,15.0497
1,hybrid_late_fusion_domain_core_light,0.1895,0.3687,0.7521,0.7905,0.8822,0.8214,0.0371,15.0394
2,hybrid_late_fusion_domain_core_base,0.1895,0.3687,0.7519,0.7903,0.8822,0.8210,0.0372,15.0342


domain text cleanup kept the current promoted hybrid: hybrid_late_fusion_custom_keep_negation_light_clean


## 16. Try Cleanup Based On Real Errors

The previous text cleanup only had a small effect on the results, so this pass uses our own mistakes like false positives and missed severe complaints to attempt another cleanup. We'll check for common text noise among our mistakes and try a small cleaning pass based on any patterns observed.

In [90]:
error_text_settings = get_text_settings_from_state(promoted_hybrid_state)
error_review_df, _, _ = build_valid_review_frame(promoted_hybrid_state['valid_score'])
error_review_df['error_slice'] = np.select(
    [
        error_review_df['urgency_band'].eq('high_urgency') & error_review_df[TARGET_COL],
        error_review_df['urgency_band'].eq('high_urgency') & ~error_review_df[TARGET_COL],
        error_review_df[TARGET_COL] & error_review_df['urgency_band'].eq('routine')
    ],
    [
        'captured_high_urgency_severe',
        'top_band_false_positive',
        'missed_severe_routine'
    ],
    default='other_valid_2025'
)

raw_error_text = error_review_df['cdescr_model_text'].fillna('').astype(str)
for flag_name, pattern in ERROR_TEXT_FLAG_PATTERNS:
    error_review_df[flag_name] = raw_error_text.str.contains(pattern, regex=True)

error_slice_summary = error_review_df.loc[
    error_review_df['error_slice'].ne('other_valid_2025')
].groupby('error_slice', observed=False).agg(
    rows=('odino', 'size'),
    primary_rate=(TARGET_COL, 'mean'),
    broad_rate=('severity_broad_flag', 'mean'),
    avg_urgency_score=('urgency_score', 'mean')
).reset_index()

error_noise_rows = []
for slice_name in ['captured_high_urgency_severe', 'top_band_false_positive', 'missed_severe_routine']:
    slice_df = error_review_df.loc[error_review_df['error_slice'].eq(slice_name)]
    if slice_df.empty:
        continue
    row = {
        'error_slice': slice_name,
        'rows': len(slice_df)
    }
    for flag_name, _ in ERROR_TEXT_FLAG_PATTERNS:
        row[flag_name] = slice_df[flag_name].mean()
    error_noise_rows.append(row)
error_noise_summary = pd.DataFrame(error_noise_rows)

error_variant_configs = []
seen_error_configs = set()
for variant_name, clean_mode in [
    ('error_driven_current_mode', error_text_settings['clean_mode']),
    ('error_driven_light_mode', 'light')
]:
    config_key = (clean_mode, error_text_settings['domain_cleanup'], True)
    if config_key in seen_error_configs:
        continue
    seen_error_configs.add(config_key)
    error_variant_configs.append({
        'name': variant_name,
        'clean_mode': clean_mode,
        'domain_cleanup': error_text_settings['domain_cleanup'],
        'error_cleanup': True
    })

print("Error slice summary on valid_2025")
display(format_display_table(
    error_slice_summary,
    rename_map={
        'error_slice': 'slice',
        'avg_urgency_score': 'avg_score'
    }
))

print("Potential text noise rates by error slice")
display(format_display_table(
    error_noise_summary,
    rename_map={
        'error_slice': 'slice',
        'has_contact_boilerplate': 'boilerplate',
        'has_redaction_artifact': 'redaction',
        'has_url_or_email': 'url_or_email',
        'has_phone_like': 'phone',
        'has_vin_like': 'vin',
        'has_case_or_claim_id': 'claim_id',
        'has_long_mixed_id': 'mixed_id'
    }
))

promoted_hybrid_state = run_text_hybrid_variant_search(
    'error-driven text cleanup',
    error_variant_configs,
    error_text_settings,
    base_structured_state,
    structured_matrix_bundle,
    promoted_hybrid_state,
    feature_block='core_structured'
)

Error slice summary on valid_2025


,slice,rows,primary_rate,broad_rate,avg_score
0,captured_high_urgency_severe,736,1.0,1.0,0.9997
1,missed_severe_routine,455,1.0,1.0,0.1901
2,top_band_false_positive,3,0.0,0.6667,0.9993


Potential text noise rates by error slice


,slice,rows,boilerplate,redaction,url_or_email,phone,vin,claim_id,mixed_id
0,captured_high_urgency_severe,736,0.5014,0.0679,0.0000,0.0027,0.0,0.0136,0.1087
1,top_band_false_positive,3,0.6667,0.0000,0.0000,0.0000,0.0,0.0000,0.0000
2,missed_severe_routine,455,0.0418,0.1011,0.0022,0.0110,0.0,0.0264,0.0418


error-driven text cleanup: text-only


,model,r@1%,r@2%,r@5%,p@5%,r@10%,pr_auc,brier,lift@5%
0,text_custom_keep_negation_light_clean,0.1885,0.3723,0.7436,0.7816,0.8712,0.8224,0.0429,14.8697
1,text_error_driven_current_mode,0.1887,0.3728,0.7434,0.7814,0.8709,0.8225,0.0429,14.8646


error-driven text cleanup: hybrid


,model,r@1%,r@2%,r@5%,p@5%,r@10%,pr_auc,brier,lift@5%
0,hybrid_late_fusion_error_driven_current_mode,0.1895,0.369,0.7531,0.7916,0.884,0.8216,0.0372,15.0599
1,hybrid_late_fusion_custom_keep_negation_light_...,0.1895,0.369,0.7526,0.7911,0.883,0.8215,0.0372,15.0497


Promoted after error-driven text cleanup: hybrid_late_fusion_error_driven_current_mode


## 17. Try Component Group Features

The severity table does not carry the normalized component groups which could be strong indicators, but the component pipeline already built the multi-label parquet that we can use. We'll test a small version of that signal to see if it's worth adding, component group count and a binary flag for each of the three highest severity components from the training data with at least 100 complaints.

In [91]:
if not COMPONENT_MULTI_PATH.exists():
    raise FileNotFoundError(f"Run consolidated preprocessing first. Missing input file: {COMPONENT_MULTI_PATH}")

component_case_df = pd.read_parquet(COMPONENT_MULTI_PATH)[['odino', 'component_groups', 'component_group_count']].copy()

train_df_component, valid_df_component, holdout_df_component, component_feature_cols, component_coverage_summary, selected_component_groups = build_component_group_feature_frames(
    train_df,
    valid_df,
    holdout_df,
    component_case_df
)

component_num_features = structured_num_features + component_feature_cols
component_structured_matrix_bundle = build_structured_matrices(
    train_df_component,
    valid_df_component,
    holdout_df_component,
    structured_cat_features,
    component_num_features
)
X_train_structured_component_matrix, X_valid_structured_component_matrix, X_holdout_structured_component_matrix = component_structured_matrix_bundle

structured_component_state = fit_linear_candidate_from_matrices(
    'structured_core_plus_top_component_groups_sgd',
    X_train_structured_component_matrix,
    X_valid_structured_component_matrix,
    X_holdout_structured_component_matrix,
    y_train,
    y_valid,
    y_holdout,
    alpha=DEFAULT_ALPHA,
    architecture='structured',
    settings={
        'feature_block': 'core_plus_top_component_groups',
        'uses_cohort_history': False,
        'uses_component_groups': True,
        'cat_features': structured_cat_features,
        'num_features': component_num_features
    }
)

structured_component_selection, structured_component_winner = choose_between_candidates(
    ['structured_core_sgd', 'structured_core_plus_top_component_groups_sgd'],
    current_name='structured_core_sgd'
)

print("Component group coverage summary")
display(format_display_table(
    component_coverage_summary,
    rename_map={
        'component_info_share': 'info_share',
        'avg_component_group_count_when_present': 'avg_group_count'
    }
))

print("Selected supported component group toggles from the training split")
display(format_display_table(
    selected_component_groups,
    rename_map={
        'component_group': 'group',
        'train_case_count': 'train_rows',
        'train_primary_rate': 'primary_rate',
        'feature_name': 'feature'
    }
))

print("Structured comparison after adding component group features")
display(compact_candidate_view(structured_component_selection))

if structured_component_winner['model'] == 'structured_core_plus_top_component_groups_sgd':
    if promoted_hybrid_state['architecture'] == 'late_fusion':
        promoted_text_state = get_state(promoted_hybrid_state['settings']['text_component_name'])
        hybrid_component_state = build_late_fusion_state(
            'hybrid_late_fusion_plus_top_component_groups',
            structured_component_state,
            promoted_text_state,
            promoted_hybrid_state['settings']['weight']
        )
    else:
        promoted_text_settings = get_text_settings_from_state(promoted_hybrid_state)
        train_text_current, valid_text_current, holdout_text_current = build_text_triplet(
            clean_mode=promoted_text_settings['clean_mode'],
            domain_cleanup=promoted_text_settings['domain_cleanup'],
            error_cleanup=promoted_text_settings['error_cleanup']
        )
        hybrid_component_state = fit_early_fusion_candidate(
            'hybrid_early_fusion_plus_top_component_groups',
            X_train_structured_component_matrix,
            X_valid_structured_component_matrix,
            X_holdout_structured_component_matrix,
            train_text_current,
            valid_text_current,
            holdout_text_current,
            y_train,
            y_valid,
            y_holdout,
            stop_words_label=promoted_text_settings['stop_words_label'],
            clean_mode=promoted_text_settings['clean_mode'],
            word_max_features=promoted_text_settings['word_max_features'],
            char_max_features=promoted_text_settings['char_max_features'],
            domain_cleanup=promoted_text_settings['domain_cleanup'],
            error_cleanup=promoted_text_settings['error_cleanup'],
            alpha=promoted_text_settings['alpha'],
            settings={
                'feature_block': 'core_plus_top_component_groups',
                'uses_cohort_history': False,
                'uses_component_groups': True
            }
        )

    hybrid_component_selection, hybrid_component_winner = choose_between_candidates(
        [promoted_hybrid_state['name'], hybrid_component_state['name']],
        current_name=promoted_hybrid_state['name']
    )
    hybrid_component_lookup = {hybrid_component_state['name']: hybrid_component_state}
    print("Hybrid comparison after adding component group features")
    display(compact_candidate_view(hybrid_component_selection))
    promoted_hybrid_state = update_promoted_state(promoted_hybrid_state, hybrid_component_winner, hybrid_component_lookup, 'adding component group features')
else:
    print("Component group features did not beat the current structured baseline, so the hybrid architecture stays unchanged")

Component group coverage summary


,split,rows,info_share,avg_group_count
0,train_2020_2024,290889,0.9226,1.352
1,valid_2025,73985,0.9196,1.390
2,holdout_2026,11063,0.9213,1.427


Selected supported component group toggles from the training split


,group,train_rows,primary_rate,feature
0,FIRERELATED,130,0.9538,cg_firerelated
1,AIR BAGS,19567,0.2253,cg_air_bags
2,SEATS,4168,0.137,cg_seats


Structured comparison after adding component group features


,model,r@1%,r@2%,r@5%,p@5%,r@10%,pr_auc,brier,lift@5%
0,structured_core_plus_top_component_groups_sgd,0.1795,0.3440,0.4520,0.4751,0.4994,0.4734,0.0824,9.0391
1,structured_core_sgd,0.1731,0.3422,0.4294,0.4514,0.4772,0.4447,0.0759,8.5866


Hybrid comparison after adding component group features


,model,r@1%,r@2%,r@5%,p@5%,r@10%,pr_auc,brier,lift@5%
0,hybrid_late_fusion_error_driven_current_mode,0.1895,0.369,0.7531,0.7916,0.8840,0.8216,0.0372,15.0599
1,hybrid_late_fusion_plus_top_component_groups,0.1898,0.368,0.7483,0.7865,0.8833,0.8193,0.0375,14.9623


adding component group features kept the current promoted hybrid: hybrid_late_fusion_error_driven_current_mode


## 18. Optional Tuning Search

This section is optional because it is slow on local hardware (~1 hour). Only run it after a cheaper experiment gives a real reason to believe that better settings are possible.

The naming conventions are getting kind of insane, but I want them to be easily trackable throughout the notebook to see what exactly was added or changed at each step.

In [92]:
RUN_BOUNDED_TUNING = False
alpha_grid = [3e-6, 1e-5, 3e-5, 1e-4]
word_feature_grid = [20000, 30000]
char_feature_grid = [10000, 20000]


def alpha_label(alpha):
    return f"{alpha:.0e}".replace('-', 'm').replace('+', '')


if not RUN_BOUNDED_TUNING:
    print("Skipped bounded tuning. Leave RUN_BOUNDED_TUNING = False until a cheap feature pass clearly beats the current promoted hybrid.")
else:
    text_settings = get_text_settings_from_state(promoted_hybrid_state)
    uses_component_groups = bool(promoted_hybrid_state['settings'].get('uses_component_groups', False))
    train_text_tune, valid_text_tune, holdout_text_tune = build_text_triplet(
        clean_mode=text_settings['clean_mode'],
        domain_cleanup=text_settings['domain_cleanup'],
        error_cleanup=text_settings['error_cleanup']
    )

    tuned_candidate_names = [promoted_hybrid_state['name']]
    structured_feature_block = 'core_plus_top_component_groups' if uses_component_groups else 'core_structured'
    structured_matrix_bundle_current = get_structured_matrix_bundle(uses_component_groups)

    if promoted_hybrid_state['architecture'] == 'late_fusion':
        structured_state_cache = {}
        base_weight = float(promoted_hybrid_state['settings']['weight'])
        tuned_weights = sorted({
            round(max(0.50, min(0.95, base_weight + delta)), 2)
            for delta in np.arange(-0.08, 0.081, 0.02)
        })

        for alpha in alpha_grid:
            if alpha not in structured_state_cache:
                structured_state_name = f"structured_tuned_{'component_groups' if uses_component_groups else 'core'}_alpha_{alpha_label(alpha)}"
                structured_state_cache[alpha] = fit_linear_candidate_from_matrices(
                    structured_state_name,
                    structured_matrix_bundle_current[0],
                    structured_matrix_bundle_current[1],
                    structured_matrix_bundle_current[2],
                    y_train,
                    y_valid,
                    y_holdout,
                    alpha=alpha,
                    architecture='structured',
                    settings={
                        'feature_block': structured_feature_block,
                        'uses_cohort_history': False,
                        'uses_component_groups': uses_component_groups,
                        'alpha': alpha
                    }
                )

            for word_max_features in word_feature_grid:
                for char_max_features in char_feature_grid:
                    text_state = fit_text_candidate(
                        f"text_tuned_alpha_{alpha_label(alpha)}_word_{word_max_features}_char_{char_max_features}",
                        train_text_tune,
                        valid_text_tune,
                        holdout_text_tune,
                        y_train,
                        y_valid,
                        y_holdout,
                        stop_words_label=text_settings['stop_words_label'],
                        clean_mode=text_settings['clean_mode'],
                        word_max_features=word_max_features,
                        char_max_features=char_max_features,
                        domain_cleanup=text_settings['domain_cleanup'],
                        error_cleanup=text_settings['error_cleanup'],
                        alpha=alpha
                    )

                    for weight in tuned_weights:
                        candidate_name = (
                            f"hybrid_late_fusion_tuned_alpha_{alpha_label(alpha)}"
                            f"_word_{word_max_features}_char_{char_max_features}_w_{weight:.2f}"
                        ).replace('.', '_')
                        tuned_state = build_late_fusion_state(
                            candidate_name,
                            structured_state_cache[alpha],
                            text_state,
                            weight
                        )
                        tuned_candidate_names.append(tuned_state['name'])
    else:
        for alpha in alpha_grid:
            for word_max_features in word_feature_grid:
                for char_max_features in char_feature_grid:
                    candidate_name = (
                        f"hybrid_early_fusion_tuned_alpha_{alpha_label(alpha)}"
                        f"_word_{word_max_features}_char_{char_max_features}"
                    )
                    tuned_state = fit_early_fusion_candidate(
                        candidate_name,
                        structured_matrix_bundle_current[0],
                        structured_matrix_bundle_current[1],
                        structured_matrix_bundle_current[2],
                        train_text_tune,
                        valid_text_tune,
                        holdout_text_tune,
                        y_train,
                        y_valid,
                        y_holdout,
                        stop_words_label=text_settings['stop_words_label'],
                        clean_mode=text_settings['clean_mode'],
                        word_max_features=word_max_features,
                        char_max_features=char_max_features,
                        domain_cleanup=text_settings['domain_cleanup'],
                        error_cleanup=text_settings['error_cleanup'],
                        alpha=alpha,
                        settings={
                            'feature_block': structured_feature_block,
                            'uses_cohort_history': False,
                            'uses_component_groups': uses_component_groups
                        }
                    )
                    tuned_candidate_names.append(tuned_state['name'])

    tuned_selection, tuned_winner = choose_between_candidates(
        tuned_candidate_names,
        current_name=promoted_hybrid_state['name']
    )
    tuned_lookup = {
        state['name']: state
        for state in state_registry.values()
        if state['name'] in tuned_selection['model'].values
    }

    display(compact_candidate_view(tuned_selection, max_rows=12))
    promoted_hybrid_state = update_promoted_state(promoted_hybrid_state, tuned_winner, tuned_lookup, 'bounded tuning')

Skipped bounded tuning. Leave RUN_BOUNDED_TUNING = False until a cheap feature pass clearly beats the current promoted hybrid.


## 19. Check Score Calibration

At this stage the model structure is fixed and we can now check whether the scores act more like sensible probabilities.

Calibration is important for this task because two models can rank complaints similarly while producing very different scores. A calibrated score is easier to read as an urgency level. Because this step is still fit on `valid_2025`, it should be treated as tuning guidance rather than the final claim.

In [93]:
CALIBRATION_SUFFIXES = ('_sigmoid_calibrated', '_isotonic_calibrated')

def strip_calibration_suffix(model_name):
    base_name = model_name
    changed = True
    while changed:
        changed = False
        for suffix in CALIBRATION_SUFFIXES:
            if base_name.endswith(suffix):
                base_name = base_name[:-len(suffix)]
                changed = True
    return base_name

def reset_calibration_artifacts(base_model_name):
    global severity_results
    stale_names = {
        model_name
        for model_name in state_registry
        if model_name != base_model_name and strip_calibration_suffix(model_name) == base_model_name
    }
    severity_results[:] = [
        row for row in severity_results
        if not (row['model'] != base_model_name and strip_calibration_suffix(row['model']) == base_model_name)
    ]
    for model_name in stale_names:
        state_registry.pop(model_name, None)

base_calibration_name = strip_calibration_suffix(promoted_hybrid_state['name'])
base_calibration_state = dict(get_state(base_calibration_name))
base_calibration_row = get_result_row(base_calibration_name)
valid_input = np.asarray(base_calibration_state['valid_calibration_input'])
holdout_input = np.asarray(base_calibration_state['holdout_calibration_input'])

sigmoid_calibrator = LogisticRegression(solver='lbfgs', max_iter=1000, random_state=RANDOM_SEED)
sigmoid_calibrator.fit(valid_input.reshape(-1, 1), y_valid)
sigmoid_valid_score = sigmoid_calibrator.predict_proba(valid_input.reshape(-1, 1))[:, 1]
sigmoid_holdout_score = sigmoid_calibrator.predict_proba(holdout_input.reshape(-1, 1))[:, 1]
sigmoid_name = f"{base_calibration_name}_sigmoid_calibrated"
isotonic_name = f"{base_calibration_name}_isotonic_calibrated"
reset_calibration_artifacts(base_calibration_name)
evaluate_scores(sigmoid_name, 'valid_2025', y_valid, sigmoid_valid_score)
register_state({
    'name': sigmoid_name,
    'architecture': base_calibration_state['architecture'],
    'model': sigmoid_calibrator,
    'valid_score': sigmoid_valid_score,
    'holdout_score': sigmoid_holdout_score,
    'valid_calibration_input': valid_input,
    'holdout_calibration_input': holdout_input,
    'settings': {**base_calibration_state['settings'], 'calibration_method': 'sigmoid', 'calibration_base_name': base_calibration_name}
})

isotonic_calibrator = IsotonicRegression(out_of_bounds='clip')
isotonic_calibrator.fit(valid_input, y_valid)
isotonic_valid_score = isotonic_calibrator.predict(valid_input)
isotonic_holdout_score = isotonic_calibrator.predict(holdout_input)
evaluate_scores(isotonic_name, 'valid_2025', y_valid, isotonic_valid_score)
register_state({
    'name': isotonic_name,
    'architecture': base_calibration_state['architecture'],
    'model': isotonic_calibrator,
    'valid_score': isotonic_valid_score,
    'holdout_score': isotonic_holdout_score,
    'valid_calibration_input': valid_input,
    'holdout_calibration_input': holdout_input,
    'settings': {**base_calibration_state['settings'], 'calibration_method': 'isotonic', 'calibration_base_name': base_calibration_name}
})

calibration_compare = build_candidate_selection_table([
    base_calibration_name,
    sigmoid_name,
    isotonic_name
])

sigmoid_row = get_result_row(sigmoid_name)
isotonic_row = get_result_row(isotonic_name)
calibration_pool = calibration_compare.loc[
    np.abs(calibration_compare['recall_top_5pct'] - base_calibration_row['recall_top_5pct']) <= 0.002
].copy()

if calibration_pool.empty:
    calibration_winner_name = base_calibration_name
else:
    calibrated_rows = calibration_pool.loc[
        calibration_pool['model'].isin([sigmoid_name, isotonic_name])
    ].copy()
    if len(calibrated_rows) == 2 and abs(sigmoid_row['brier_score'] - isotonic_row['brier_score']) <= 0.002:
        preferred_calibrated_name = sigmoid_name
    elif not calibrated_rows.empty:
        preferred_calibrated_name = calibrated_rows.sort_values('brier_score').iloc[0]['model']
    else:
        preferred_calibrated_name = None

    if preferred_calibrated_name is None:
        calibration_winner_name = base_calibration_name
    else:
        preferred_calibrated_row = get_result_row(preferred_calibrated_name)
        calibration_winner_name = preferred_calibrated_name if preferred_calibrated_row['brier_score'] < base_calibration_row['brier_score'] else base_calibration_name

print("Calibration comparison")
display(compact_candidate_view(calibration_compare))

if calibration_winner_name != base_calibration_name:
    promoted_hybrid_state = dict(get_state(calibration_winner_name))
    print("Promoted after calibration:", promoted_hybrid_state['name'])
else:
    promoted_hybrid_state = dict(base_calibration_state)
    print("Raw promoted hybrid stayed ahead after the calibration guardrails")

Calibration comparison


,model,r@1%,r@2%,r@5%,p@5%,r@10%,pr_auc,brier,lift@5%
0,hybrid_late_fusion_error_driven_current_mode_i...,0.1898,0.369,0.7537,0.7922,0.8843,0.8132,0.0179,15.0702
1,hybrid_late_fusion_error_driven_current_mode_s...,0.1895,0.369,0.7531,0.7916,0.8840,0.8216,0.0185,15.0599
2,hybrid_late_fusion_error_driven_current_mode,0.1895,0.369,0.7531,0.7916,0.8840,0.8216,0.0372,15.0599


Promoted after calibration: hybrid_late_fusion_error_driven_current_mode_sigmoid_calibrated


## 20. Summary Of The Best Model

Here we have the final summary of the current best notebook model, it gives the settings we settled on at each major experiment as well as the metrics for the chosen model. We then look at a few readouts that would be important for a reviewer: the review budget table, the urgency band breakdowns, and the highest scored complaints.

In [94]:
promoted_row = get_result_row(promoted_hybrid_state['name'])
promoted_operating_table = build_operating_point_table(
    promoted_hybrid_state['name'],
    'valid_2025',
    y_valid,
    promoted_hybrid_state['valid_score']
)

promoted_valid_df, _, review_cols = build_valid_review_frame(promoted_hybrid_state['valid_score'])
promoted_band_summary, promoted_monthly_burden = summarize_review_frame(promoted_valid_df)
promoted_top_risk = promoted_valid_df.sort_values('urgency_score', ascending=False)[review_cols].head(10)

print("Promoted candidate:", promoted_hybrid_state['name'])
print("Architecture:", promoted_hybrid_state['architecture'])
print("Settings:")
display(pd.Series(promoted_hybrid_state['settings'], name='value').rename_axis('setting').to_frame())

print("Promoted model row on valid_2025")
display(compact_result_view(promoted_row))

print("Promoted operating points on valid_2025")
display(compact_budget_view(promoted_operating_table))

print("Promoted urgency band counts on valid_2025")
display(promoted_band_summary)

print("Promoted monthly review burden on valid_2025")
display(promoted_monthly_burden)

print("Top 10 complaints under the promoted urgency rule")
display(promoted_top_risk)

Promoted candidate: hybrid_late_fusion_error_driven_current_mode_sigmoid_calibrated
Architecture: late_fusion
Settings:


,value
setting,
weight,0.75
text_component_name,text_error_driven_current_mode
structured_component_name,structured_core_sgd
uses_cohort_history,False
uses_component_groups,False
calibration_method,sigmoid
calibration_base_name,hybrid_late_fusion_error_driven_current_mode


Promoted model row on valid_2025


,model,split,base_rate,pr_auc,roc_auc,f1,precision,recall,brier,r@1%,r@2%,r@5%,p@5%,lift@5%,r@10%,p@10%,lift@10%
13,hybrid_late_fusion_error_driven_current_mode_s...,valid_2025,0.052565,0.821617,0.961671,0.771448,0.818523,0.729493,0.018543,0.189509,0.368989,0.75315,0.791622,15.059945,0.884032,0.464657,8.839721


Promoted operating points on valid_2025


...

,urgency_band,rows,share
0,high_urgency,739,0.009989
1,priority_review,2960,0.040008
2,watch,3699,0.049997
3,routine,66587,0.900007


Promoted monthly review burden on valid_2025


urgency_band,review_month,high_urgency,priority_review,watch,routine
0,2025-01,43,230,301,5910
1,2025-02,53,182,268,5089
2,2025-03,51,210,287,5624
3,2025-04,62,167,278,5565
4,2025-05,57,214,290,5456
5,2025-06,45,227,284,5654
6,2025-07,63,307,347,6516
7,2025-08,75,293,332,5798
8,2025-09,79,306,334,5640
9,2025-10,75,283,343,5452


Top 10 complaints under the promoted urgency rule


,odino,ldate,mfr_name,maketxt,modeltxt,yeartxt,urgency_score,urgency_band,suspicious_value_any,severity_primary_flag,severity_broad_flag,flag_speed_999,flag_speed_high,flag_miles_high,flag_injured_99,flag_deaths_99,cdescr_model_text
295452,11638081,2025-01-23,"BMW OF NORTH AMERICA, LLC",MINI,COOPER,2009,0.950768,high_urgency,False,True,True,False,False,False,False,False,The contact owned a 2009 Mini Cooper. The cont...
361453,11704694,2025-12-12,"SUBARU OF AMERICA, INC.",SUBARU,FORESTER,2019,0.950768,high_urgency,False,True,True,False,False,False,False,False,The contact owned a 2019 Subaru Forester. The ...
342897,11685973,2025-09-09,"CHRYSLER (FCA US, LLC)",RAM,1500,2021,0.950768,high_urgency,False,True,True,False,False,False,False,False,The contact owned a 2021 RAM 1500. The contact...
352572,11695774,2025-10-27,"TESLA, INC.",TESLA,MODEL Y,2026,0.950767,high_urgency,False,True,True,False,False,False,False,False,The contact owned a 2026 Tesla Model Y. The co...
339320,11682374,2025-08-22,"MERCEDES-BENZ USA, LLC",MERCEDES-BENZ,GLC 300,2019,0.950767,high_urgency,False,True,True,False,False,False,False,False,The contact owned a 2019 Mercedes-Benz GLC 300...
321035,11663963,2025-05-30,TOYOTA MOTOR CORPORATION,TOYOTA,RAV4,2020,0.950767,high_urgency,False,True,True,False,False,False,False,False,The contact owns a 2020 Toyota RAV4. The conta...
347888,11691052,2025-10-02,FORD MOTOR COMPANY,FORD,BRONCO SPORT,2023,0.950767,high_urgency,False,True,True,False,False,False,False,False,The contact owned a 2023 Ford Bronco Sport. Th...
358046,11701277,2025-11-24,FORD MOTOR COMPANY,MERCURY,SABLE,2008,0.950767,high_urgency,False,True,True,False,False,False,False,False,The contact owns a 2008 Mercury Sable. While t...
315041,11657891,2025-04-30,FORD MOTOR COMPANY,FORD,FIESTA,2014,0.950767,high_urgency,False,True,True,False,False,False,False,False,I was involved in a hit and run accident where...
351993,11695192,2025-10-23,"GENERAL MOTORS, LLC",CADILLAC,CTS,2007,0.950767,high_urgency,False,True,True,False,False,False,False,False,The contact owned a 2007 Cadillac CTS. The con...


## 21. Optional Final 2026 Check

Leave this off during normal experimentation. Turn it on only when you want one reference only read on 2026 for the current best model.

In [95]:
RUN_HOLDOUT = False

if RUN_HOLDOUT:
    evaluate_scores(promoted_hybrid_state['name'], 'holdout_2026', y_holdout, promoted_hybrid_state['holdout_score'])
    holdout_row = get_result_row(promoted_hybrid_state['name'], split_name='holdout_2026')
    print("Soft-final holdout reference")
    display(compact_result_view(holdout_row))
else:
    print("Holdout skipped. Leave RUN_HOLDOUT = False until you want one soft-final 2026 reference readout for the promoted hybrid rule.")

Holdout skipped. Leave RUN_HOLDOUT = False until you want one soft-final 2026 reference readout for the promoted hybrid rule.


## 22. Save Notebook Results

Write the compact results table so the current notebook state can be reviewed outside the notebook.


In [96]:
results_df = show_results(display_table=False)
results_df.to_csv(SEVERITY_RESULTS_PATH, index=False)
print("Wrote:", SEVERITY_RESULTS_PATH)

Wrote: C:\Users\davis\OneDrive\Documents\VS Code Repos\NHTSA-ODI-Complaint-Analytics\data\outputs\severity_partner_results.csv


## 23. What To Try Next

The tuned, sigmoid-calibrated late-fusion hybrid is the current notebook benchmark.

Quick summary of what notebook experimentation showed:
- small text cleanup passes were worth doing and improved the urgency rule
- component group features helped the structured only model but did not improve the final hybrid queue
- model promotion is still based on top 5% review performance, not only on a global ranking metric

The only remaining real logic step I may do is a `severity_broad_flag` sensitivity run, otherwise I'll like just do some formatting and readability cleanup then add this rules to the pipeline.